In [ ]:
# Getting relevant libraries

import numpy as np 
import random
import matplotlib.pyplot as plt
import math
import matplotlib.cm as cm
import pickle
import os
import pandas as pd
import random
plt.rcParams['figure.figsize'] = [10, 7]
from matplotlib.colors import LinearSegmentedColormap
#import mpl_scatter_density # adds projection='scatter_density'
from scipy.stats import gaussian_kde
from scipy import optimize
from molmass import Formula
import csv
import re
import copy
import gc
import time
import molmass as ms
from tqdm import tqdm
from BackEnds.EmulatorLibrary import *
#from BackEnds.nnMELTS import DualSaturationChemistry, TunableModel, MidLevelNetwork
import BackEnds.nnMELTS as NN
import importlib 
from BackEnds.nnMELTS import rebuild_MELTS_model 

In [ ]:
"""TORCH ML LOADING. MUST HAPPEN AFTER ABOVE BLOCK IS RUN (for some reason...)"""
import torch
from torch.utils.data import DataLoader, Dataset, random_split
from torch.autograd import Variable
from torch.nn import Linear, ReLU, CrossEntropyLoss, Sequential, Conv2d, MaxPool2d, Module, Softmax, Dropout, BCELoss, Sigmoid, MSELoss
from torch.optim import Adam, SGD, AdamW
import torch.nn as nn
import torch.nn.functional as F
from BackEnds.SaturationDataset import TensorDatasetNormalized, TensorDataset, TensorDatasetThree, TensorDatasetFour

In [ ]:




def relative_L1_loss(y_pred, y_true, mask=None, eps=1e-6):
    rel_error = (y_pred - y_true).abs() / (y_true.abs() + eps)
    if mask is not None:
        rel_error = rel_error * mask
        return rel_error.sum() / mask.sum().clamp(min=1)
    return rel_error.mean()

def symmetric_rel_l1(pred, target, eps=1e-6):
    denom = torch.clamp(torch.abs(pred) + torch.abs(target), min=eps)
    return torch.mean(torch.abs(pred - target) / denom)

def symmetric_rel_l2(pred, target, eps=1e-6):
    denom = torch.clamp(torch.abs(pred) + torch.abs(target), min=eps)
    return torch.mean((pred - target)**2 / denom)


    

In [ ]:
#time.sleep(3600*3)
MELTSModel= '102'
CalcType = 'FxCryst'
date = 'Nov9'
use_external = True # Is data on external drive? Path defined in EmulatorLibrary.py
subset = False

Trainfilename = f'{internal_dir(MELTSModel)}MELTS{MELTSModel}_Trainset{date}{CalcType}Cooling'
Testfilename = f'{internal_dir(MELTSModel)}MELTS{MELTSModel}_Testset{date}{CalcType}Cooling'
modelname =f"MELTS{MELTSModel}{CalcType}"

if subset:
    Trainfilename += '_subset'
    Testfilename += '_subset'

if not os.path.exists(f'{Trainfilename}features.npy') or use_external == True:
    Trainfilename = external_base + Trainfilename
if not os.path.exists(f'{Testfilename}features.npy') or use_external == True:
    Testfilename = external_base + Testfilename


if not os.path.exists(f"Models/"):
    os.makedirs('Models/')

#time.sleep(3600) # hr delay for data processing, add another five minutes to this time 

PTfO2min = torch.tensor([1,700,-5], device = 'cpu', dtype = torch.float)
PTfO2max = torch.tensor([10000,2000,5], device = 'cpu', dtype = torch.float)
min_tensor = torch.zeros(len(Elkeys)+3, device = 'cpu', dtype = torch.float)
min_tensor[:3] = PTfO2min
range_tensor = torch.ones(len(Elkeys)+3, device = 'cpu', dtype = torch.float)
range_tensor[:3] = PTfO2max - PTfO2min



class Normalizer:
    """Quick Normalzing object that holds minima and ranges for a dataset and converts into and out of [0,1]
    minmax normalization for interfacing with neural networks"""
    
    def __init__(self, min_tensor, range_tensor):
        assert len(min_tensor) == len(range_tensor), 'Minimum and range are not equal!'
        self.miner = min_tensor
        self.ranger = range_tensor
        
    def __len__(self):
        return len(self.miner)

    def denorm(self, x):
        return x * self.ranger + self.miner
    
    def norm(self, x):
        return (x - self.miner) / self.ranger



feature_path = Trainfilename+'features.npy'
binary_path  = Trainfilename+'binary_labels.npy'
label_path = Trainfilename+'labels.npy'
mole_path = Trainfilename+'molar_labels.npy'

featureMap = np.load(feature_path, mmap_mode='r')
binaryMap = np.load(binary_path, mmap_mode='r')
labelMap = np.load(label_path, mmap_mode='r')
moleMap = np.load(mole_path, mmap_mode='r')

print(f"Feature Shape: {featureMap.shape}")
print(f"Binary Shape: {binaryMap.shape}")
print(f"Label Shape: {labelMap.shape}")
print(f"Mole Shape: {moleMap.shape}")



min_tensor = torch.zeros(featureMap.shape[1], device = 'cpu', dtype = torch.float)
min_tensor[:3] = PTfO2min
range_tensor = torch.ones(featureMap.shape[1], device = 'cpu', dtype = torch.float)
range_tensor[:3] = PTfO2max - PTfO2min
normf = Normalizer(min_tensor=min_tensor, range_tensor=range_tensor)

Trainnormfeatures = normf.norm(torch.tensor(featureMap, device = 'cpu', dtype = torch.float))
del featureMap
gc.collect()

Trainbinaryfeatures = torch.tensor(binaryMap, device = 'cpu', dtype = torch.float)
del binaryMap
gc.collect()

Trainlabels = torch.tensor(labelMap, device = 'cpu', dtype = torch.float) @ torch.tensor(PxSpTransform[np.ix_(compositional_component_subset, compositional_component_subset)], dtype = torch.float)
del labelMap
gc.collect()

Trainmoles = torch.tensor(moleMap, device = 'cpu', dtype=torch.float).detach().numpy()
del moleMap
gc.collect()


def process_in_batches(Trainnormfeatures, Trainmoles, Trainlabels, batch_size=8192):

    # Precompute constant matrices as float32 tensors
    oxToEl_t = torch.tensor(oxToEl[:-1], dtype=torch.float32)
    MM_t = torch.tensor(MM[:-1, :-1], dtype=torch.float32)
    compToOx_t = torch.tensor(compToOx, dtype=torch.float32)
    oxToEl_full_t = torch.tensor(oxToEl, dtype=torch.float32)

    # Inverse only once
    oxToEl_inv = torch.linalg.inv(oxToEl_t)

    n_samples = Trainnormfeatures.size(0)

    bulk_wt_ox_chunks = []
    GTReconBulk_chunks = []

    for start in tqdm(range(0, n_samples, batch_size)):
        end = min(start + batch_size, n_samples)

        # === Bulk weights ===
        bulk_wt_ox = (
            (Trainnormfeatures[start:end, 3:] @ oxToEl_inv) @ MM_t
        )
        bulk_wt_ox = 100 * bulk_wt_ox / torch.sum(bulk_wt_ox, axis=1).reshape(-1, 1)
        bulk_wt_ox_chunks.append(bulk_wt_ox)

        # === Ground truth compositions ===
        GT_comps = torch.zeros(
            (end - start, label_indices['melts-liquid'][-1] + 1),
            dtype=torch.float32,
        )

        for phase in np.array(list(label_indices.keys())):
            moles = torch.tensor(
                Trainmoles[start:end, mass_phasedict[phase]].reshape(-1, 1),
                dtype=torch.float32,
            )
            if phase in compositionally_variable_phases:
                GT_comps[:, label_indices[phase]] = (
                    moles * Trainlabels[start:end, label_indices_comp[phase]].to(torch.float32)
                )
            else:
                GT_comps[:, label_indices[phase]] = moles

        # === Recon bulk oxides ===
        GTReconBulk_oxides = (
            ((GT_comps @ compToOx_t) @ oxToEl_full_t) @ oxToEl_inv
        ) @ MM_t
        GTReconBulk_oxides *= 100 / torch.sum(GTReconBulk_oxides, axis=1, keepdims=True)

        GTReconBulk_chunks.append(GTReconBulk_oxides)

    # Recombine all batches
    bulk_wt_ox = torch.cat(bulk_wt_ox_chunks, dim=0)
    GTReconBulk_oxides = torch.cat(GTReconBulk_chunks, dim=0)

    # === Compare rounded results ===
    train_mismatches = torch.unique(
        torch.where(
            torch.round(bulk_wt_ox, decimals=2) != torch.round(GTReconBulk_oxides, decimals=2)
        )[0]
    )

    return train_mismatches

train_mismatches = process_in_batches(Trainnormfeatures, Trainmoles, Trainlabels, batch_size=2**13)


print(train_mismatches.size())
#assert mismatches.size()[0] == 0

OOB = ((Trainlabels > 1).to(float) + (Trainlabels < 0).to(float)).to(bool)
badMap = torch.unique(torch.where(OOB)[0])
goodMap = torch.ones(Trainlabels.size()[0]).to(torch.bool)
#goodMap = torch.arange(Testlabels.size()[0])
#goodMap = goodMap[~torch.isin(goodMap, badMap)] # Exclude OOB IDs
goodMap[badMap] = False
goodMap[train_mismatches] = False


print(f"Train Features: {Trainnormfeatures.size()}, Binaries {Trainbinaryfeatures.size()}, labels: {Trainlabels.size()}")
Trainnormfeatures, Trainbinaryfeatures, Trainlabels, Trainmoles = Trainnormfeatures[goodMap], Trainbinaryfeatures[goodMap], Trainlabels[goodMap], Trainmoles[goodMap]
print(f"Train Features: {Trainnormfeatures.size()}, Binaries {Trainbinaryfeatures.size()}, labels: {Trainlabels.size()}")

# IncludeCr free assemblages with rare phases
Cr_in = Trainnormfeatures[:,-1] != 0 + torch.any(
    Trainbinaryfeatures[:,torch.tensor([mass_phasedict[phase] for phase in ['nepheline', 'leucite', 'analcime', 'alloy-solid', 'muscovite', 'k-feldspar']])] > 0.5, 
    dim = -1).to(torch.bool)
rhm_idx = torch.where(Trainbinaryfeatures[:,mass_phasedict['rhm-oxide']] > 0.5)[0]
Cr_in[rhm_idx[torch.randperm(len(rhm_idx))[:(len(rhm_idx)//3)]]] = True # Add back 1/3rd of rhm-oxides. 

Cr_out = (Trainnormfeatures[:,-1] == 0).to(torch.bool) 
    

print(f"Chrome in Training: {Cr_in.sum()}, Chrome Absent in Training: {Cr_out.sum()}")

binary_train_set_Cr = TensorDataset(features=Trainnormfeatures[Cr_in], labels=Trainbinaryfeatures[Cr_in])
full_train_set_Cr = TensorDatasetFour(features=Trainnormfeatures[Cr_in], binarylabels=Trainbinaryfeatures[Cr_in], labels = Trainlabels[Cr_in], molelabels = Trainmoles[Cr_in])

binary_train_set_NoCr = TensorDataset(features=Trainnormfeatures[Cr_out], labels=Trainbinaryfeatures[Cr_out])
full_train_set_NoCr = TensorDatasetFour(features=Trainnormfeatures[Cr_out], binarylabels=Trainbinaryfeatures[Cr_out], labels = Trainlabels[Cr_out], molelabels = Trainmoles[Cr_out])


feature_path = Testfilename+'features.npy'
binary_path  = Testfilename+'binary_labels.npy'
label_path = Testfilename+'labels.npy'
mole_path = Testfilename+'molar_labels.npy'

featureMap = np.load(feature_path, mmap_mode='r')
binaryMap = np.load(binary_path, mmap_mode='r')
labelMap = np.load(label_path, mmap_mode='r')
moleMap = np.load(mole_path, mmap_mode ='r')

Testnormfeatures = normf.norm(torch.tensor(featureMap, device = 'cpu', dtype = torch.float))
del featureMap
gc.collect()

Testbinaryfeatures = torch.tensor(binaryMap, device = 'cpu', dtype = torch.float)
del binaryMap
gc.collect()

Testlabels = torch.tensor(labelMap, device = 'cpu', dtype = torch.float) @ torch.tensor(PxSpTransform[np.ix_(compositional_component_subset, compositional_component_subset)], dtype = torch.float)
del labelMap
gc.collect()

Testmoles = torch.tensor(moleMap, device = 'cpu', dtype=torch.float).detach().numpy()
del moleMap
gc.collect()


## --- Test split ---
bulk_wt_ox = (
    (Testnormfeatures[:, 3:] @ torch.linalg.inv(torch.tensor(oxToEl[:-1], dtype=torch.float32)))
    @ torch.tensor(MM[:-1, :-1], dtype=torch.float32)
)
bulk_wt_ox = 100 * bulk_wt_ox / torch.sum(bulk_wt_ox, axis=1).reshape(-1, 1)

GT_comps = torch.zeros(
    (Testnormfeatures.size()[0], label_indices['melts-liquid'][-1] + 1),
    dtype=torch.float32,
)

for phase in np.array(list(label_indices.keys())):
    if phase in compositionally_variable_phases:
        GT_comps[:, label_indices[phase]] = ((
            torch.tensor(Testmoles[:, mass_phasedict[phase]].reshape(-1, 1), dtype = torch.float32))
            * Testlabels[:, label_indices_comp[phase]].to(torch.float32)
        )
    else:
        GT_comps[:, label_indices[phase]] = ((
            torch.tensor(Testmoles[:, mass_phasedict[phase]].reshape(-1, 1), dtype = torch.float32))
        )

GTReconBulk_oxides = (
    ((GT_comps @ torch.tensor(compToOx, dtype=torch.float32))
     @ torch.tensor(oxToEl, dtype=torch.float32))
    @ torch.linalg.inv(torch.tensor(oxToEl[:-1], dtype=torch.float32))
) @ torch.tensor(MM[:-1, :-1], dtype=torch.float32)
GTReconBulk_oxides *= 100 / torch.sum(GTReconBulk_oxides, axis=1, keepdims=True)

test_mismatches = torch.unique(
    torch.where(torch.round(bulk_wt_ox, decimals = 2) != torch.round(GTReconBulk_oxides, decimals = 2))[0]
)
print(test_mismatches.size())
print(bulk_wt_ox.size())
#assert mismatches.size()[0] == 0, f'mismatch: {mismatches.size()[0]} out of bulk_wt_ox.size()[0]'




OOB = ((Testlabels > 1).to(float) + (Testlabels < 0).to(float)).to(bool)
badMap = torch.unique(torch.where(OOB)[0])
goodMap = torch.ones(Testlabels.size()[0]).to(torch.bool)
goodMap[badMap] = False
goodMap[test_mismatches] = False
print(f"Test Features: {Testnormfeatures.size()}, Binaries {Testbinaryfeatures.size()}, labels: {Testlabels.size()}")
Testnormfeatures, Testbinaryfeatures, Testlabels, Testmoles = Testnormfeatures[goodMap], Testbinaryfeatures[goodMap], Testlabels[goodMap], Testmoles[goodMap]
print(f"Test Features: {Testnormfeatures.size()}, Binaries {Testbinaryfeatures.size()}, labels: {Testlabels.size()}")

# IncludeCr free assemblages with rare phases
Cr_in = Testnormfeatures[:,-1] != 0 + torch.any(
    Testbinaryfeatures[:,torch.tensor([mass_phasedict[phase] for phase in ['nepheline', 'leucite', 'analcime', 'alloy-solid', 'muscovite', 'k-feldspar']])] > 0.5, 
    dim = -1).to(torch.bool)
rhm_idx = torch.where(Testbinaryfeatures[:,mass_phasedict['rhm-oxide']] > 0.5)[0]
Cr_in[rhm_idx[torch.randperm(len(rhm_idx))[:(len(rhm_idx)//3)]]] = True # Add back 1/3rd of rhm-oxides. 

Cr_out = (Testnormfeatures[:,-1] == 0).to(torch.bool) 
          
          
print(f"Chrome in Test: {Cr_in.sum()}, Chrome Absent in Test: {Cr_out.sum()}")





binary_test_set_Cr = TensorDataset(features=Testnormfeatures[Cr_in], labels=Testbinaryfeatures[Cr_in])
full_test_set_Cr = TensorDatasetFour(features=Testnormfeatures[Cr_in], binarylabels=Testbinaryfeatures[Cr_in], labels = Testlabels[Cr_in], molelabels = Testmoles[Cr_in])

binary_test_set_NoCr = TensorDataset(features=Testnormfeatures[Cr_out], labels=Testbinaryfeatures[Cr_out])
full_test_set_NoCr = TensorDatasetFour(features=Testnormfeatures[Cr_out], binarylabels=Testbinaryfeatures[Cr_out], labels = Testlabels[Cr_out], molelabels = Testmoles[Cr_out])







In [ ]:
from BackEnds.nnMELTS import MidLevelNetwork



def train_Lower_MELTS(Model, criterion = nn.BCEWithLogitsLoss(), Cr = False, Epochs = 20, lr = 1E-3, device = 'cuda'):
        # --- Build (copy) model ---
    model = NN.MidLevelNetwork(**Model.config).to(device)#.to(Model.device) # Copy the old model, so no overwriting. 

    noise = model.noise

    # freeze chem & mole heads like original code
    for p in model.parameters():
        p.requires_grad = True
    for p in model.chem_heads.parameters():
        p.requires_grad = False
    for p in model.mole_head.parameters():
        p.requires_grad = False

    if Cr:
        binary_train_set, binary_test_set = binary_train_set_Cr, binary_test_set_Cr
        print('Loading Cr')
    else:
        binary_train_set, binary_test_set = binary_train_set_NoCr, binary_test_set_NoCr
        print('Loading Non-Cr')

    if 'dropout' in model.config['low_regularization'].lower():
        dropout_rate = pull_number(model.config['low_regularization'].lower())
        print(f"dropout in {model.config['low_regularization']}: {pull_number(model.config['low_regularization'])}")
    else:
        dropout_rate = 0

    # --- Loaders (same for both "Cr" and "NoCr" if you only want one test here) ---
    batch_size = 1024
    train_loader = DataLoader(binary_train_set, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    test_loader = DataLoader(binary_test_set, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=model.config['lowWD'])
    best_test_loss = np.inf
    train_losses, test_losses = [], []

    last_missed = False
    # --- Train for specified epochs ---
    for epoch in range(Epochs):
        start = time.time()
        model.train()
        running_train_loss = 0.0

        for xb, yb in tqdm(train_loader, desc=f"Train Epoch {epoch+1}", leave=False):
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            #bulk_zero_mask = (x_batch != 0).to(torch.float) # Bulk zero mask different shape for training and testing because this mask is doubling as a filter for the noise
            if noise != 0:
                xb = xb + (xb * torch.randn_like(xb) * noise)  # noise injection

            optimizer.zero_grad()
            logits = model.forward_binaries(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_train_loss += loss.item() * xb.size(0)

        avg_train_loss = running_train_loss / len(train_loader.dataset)
        train_losses.append(avg_train_loss)

        # --- Evaluate ---
        model.eval()
        running_test_loss = 0.0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
                logits = model.forward_binaries(xb)
                loss = criterion(logits, yb)
                running_test_loss += loss.item() * xb.size(0)

        avg_test_loss = running_test_loss / len(test_loader.dataset)
        test_losses.append(avg_test_loss)

        print(f"Epoch {epoch+1:02d}: Train {avg_train_loss:.5f} | Test {avg_test_loss:.5f} | Δt={time.time()-start:.1f}s")

        # simple early stopping
        if avg_test_loss < best_test_loss:
            best_test_loss = avg_test_loss
            torch.save(model.state_dict(), 'Models/temp_binary_train.pt')
            last_missed = False
        elif epoch > 2 and avg_test_loss > best_test_loss * 1.01:
            print("No improvement; stopping early?")
            if last_missed:
                break
            last_missed = True

        #ADAPTIVE DROPOUT
        if avg_test_loss > avg_train_loss * 1.02:
            anyDropout = False
            if min(dropout_rate + 0.05, 0.6) > dropout_rate:
                old_drop = dropout_rate
                dropout_rate = min(dropout_rate + 0.05, 0.6)
                for module in model.modules():
                    if isinstance(module, nn.Dropout):
                        module.p = dropout_rate
                        anyDropout  = True
                if anyDropout:
                    print(f"Overfitting. Increasing Dropout: {old_drop} -> {dropout_rate}")
            else: 
                print(f"Overfitting, but dropout_rate is at the maximum: {dropout_rate}")


        elif avg_test_loss < avg_train_loss:
            anyDropout = False
            if max(dropout_rate - 0.02, 0) < dropout_rate:
                old_drop = dropout_rate
                dropout_rate = max(dropout_rate - 0.02, 0) 
                for module in model.modules():
                    if isinstance(module, nn.Dropout):
                        module.p = dropout_rate
                        anyDropout  = True
                if anyDropout:
                    print(f"Underfitting. Decreasing Dropout: {old_drop}->{dropout_rate}")

            else:
                print(f"Underfitting, but dropout_rate is at the minimum: {dropout_rate}")

        gc.collect()

        

    model.load_state_dict(torch.load('Models/temp_binary_train.pt', weights_only=False))
    return model, best_test_loss


weight_dict_both = {
    'olivine': 3,
    'nepheline': 15,
    'leucite': 10,
    'alloy-solid': 5,
    'muscovite': 3,
    'k-feldspar': 5
}
# This overprints the above weights, does not apply to NoCr model
wweight_dict_Cr = {
    'rhm-oxide': 5
}

binWeightsNoCr = torch.ones(len(list(label_indices.keys())), dtype = torch.float32).reshape(1,-1)
binWeightsCr = torch.ones(len(list(label_indices.keys())), dtype = torch.float32).reshape(1,-1)
compWeightsNoCr = torch.ones(label_indices_comp['melts-liquid'][-1]+1, dtype = torch.float32).reshape(1,-1)
compWeightsCr = torch.ones(label_indices_comp['melts-liquid'][-1]+1, dtype = torch.float32).reshape(1,-1)
"""for phase, W in weight_dict_both.items(): # Weights removed for 110 due to bad overfitting... Take a look at phase abundances??? 
    binWeightsCr[:,mass_phasedict[phase]] = W
    binWeightsNoCr[:,mass_phasedict[phase]] = W
    if phase in compositionally_variable_phases:
        compWeightsCr[:,comp_phasedict[phase]] = W
        compWeightsNoCr[:,comp_phasedict[phase]] = W
for phase, W in weight_dict_Cr.items():
    binWeightsCr[:,mass_phasedict[phase]] = W
    if phase in compositionally_variable_phases:
        compWeightsCr[:,comp_phasedict[phase]] = W"""


def train_Upper_MELTS(Model, criterion = symmetric_rel_l2, criterion_sat = nn.BCEWithLogitsLoss(), Cr = False, Epochs = 20, lr = 1E-3, device = 'cuda'):
        # --- Build (copy) model ---
    model = NN.MidLevelNetwork(**Model.config).to(device)#.to(Model.device) # Copy the old model, so no overwriting. 

    noise = model.noise

    # Copy Lower Parameters! 
    model.sat_head.load_state_dict(Model.sat_head.state_dict())
    model.encoder.load_state_dict(Model.encoder.state_dict())

    # freeze chem & mole heads like original code
    for p in model.parameters():
        p.requires_grad = True
    for p in model.sat_head.parameters():
        p.requires_grad = False
    for p in model.encoder.parameters():
        p.requires_grad = False
    

    if Cr:
        full_train_set, full_test_set = full_train_set_Cr, full_test_set_Cr
        print('Loading Cr')
        binWeights, compWeights = binWeightsCr, compWeightsCr
    else:
        full_train_set, full_test_set = full_train_set_NoCr, full_test_set_NoCr
        print('Loading Non-Cr')
        binWeights, compWeights = binWeightsNoCr, compWeightsNoCr

    if 'dropout' in model.config['high_regularization'].lower():
        dropout_rate = pull_number(model.config['high_regularization'].lower())
        print(f"dropout in {model.config['high_regularization']}: {pull_number(model.config['high_regularization'])}")
    else:
        dropout_rate = 0

    # --- Loaders (same for both "Cr" and "NoCr" if you only want one test here) ---
    batch_size = 1024
    train_loader = DataLoader(full_train_set, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    test_loader = DataLoader(full_test_set, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=model.config['highWD'])
    best_test_loss = np.inf
    train_losses, test_losses = [], []

    binWeights = binWeights.cuda()
    compWeights = compWeights.cuda()

    chem_alpha = 1
    mole_alpha = 1
    bulk_alpha = 0

    criterion_chem = criterion 
    criterion_mole = criterion
    criterion_bulk = criterion

    last_missed = False
    # --- Train for specified epochs ---
    for epoch in range(Epochs):
        start = time.time()
        model.train()
        running_train_loss = 0.0
        running_sat_loss = 0
        running_chem_loss = 0
        running_mole_loss = 0
        running_bulk_loss = 0
        for batch_idx, (x_batch, b_batch, y_batch, m_batch) in enumerate(tqdm(train_loader, desc="Training", leave=False)):

            optimizer.zero_grad()

            x_batch, b_batch, y_batch, m_batch = x_batch.to(device, non_blocking=True), b_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True), m_batch.to(device, non_blocking=True)
            
            bulk_zero_mask = (x_batch != 0).to(torch.float) # Bulk zero mask different shape for training and testing because this mask is doubling as a filter for the noise
            #x_batch = x_batch + torch.randn_like(x_batch) * 0.005 * bulk_zero_mask # Add Small Gaussian Noise to avoid overfitting during training Oct 14: Moved 0.002->0.005
            if noise != 0:
                x_batch = x_batch + (x_batch * torch.randn_like(x_batch) * noise)
            logits, chem_preds, chem_zero_mask, mole_preds, bulk_preds = model(x_batch, binaries=b_batch, NN_only = True)

            # Binary saturation loss
            loss_sat = criterion_sat(logits, b_batch) 

            # Chemistry and mass losses: apply masks
            chem_loss_raw = criterion_chem(chem_preds, y_batch)
            mole_loss_raw = criterion_mole(mole_preds, m_batch)
            bulk_loss_raw = criterion_bulk(bulk_preds, x_batch[:,3:])
            #mole_zero_mask = (torch.sigmoid(logits) > 0.5).to(torch.float).detach() # Only use binary preds for masking when those neurons are free
            mole_zero_mask = (b_batch > 0.5).to(torch.float).detach()

            """if batch_idx == 0:
                print(chem_loss_raw.device)
                print(chem_zero_mask.device)
                print(compWeights.device)"""

            chem_loss_masked = (chem_loss_raw * chem_zero_mask * compWeights).sum() / (chem_zero_mask * compWeights).sum().clamp(min=1)
            mole_loss_masked = (mole_loss_raw * mole_zero_mask * binWeights).sum() / (mole_zero_mask * binWeights).sum().clamp(min=1)
            bulk_loss_masked = (bulk_loss_raw * bulk_zero_mask[:,3:]).sum() / (bulk_zero_mask[:,3:]).sum().clamp(min=1)
            
            running_sat_loss += loss_sat.item()
            running_mole_loss += mole_loss_masked.item()
            running_chem_loss += chem_loss_masked.item()
            running_bulk_loss += bulk_loss_masked.item()
            
            
            
            loss = loss_sat + chem_alpha*chem_loss_masked + mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
            #print(f"Sat loss: {loss_sat.item()}, Chem loss: {chem_loss_masked.item()}")

            if not torch.isfinite(loss):
                print("Non-finite loss detected!")
                print(f"Sat loss: {loss_sat.item()}, Chem loss: {chem_loss_masked.item()}, Mole loss: {mole_loss_masked}, Bulk loss: {bulk_loss_masked}")
                continue

            loss.backward()


            optimizer.step()

            running_train_loss += loss.item() * x_batch.size(0)

            #if batch_idx % 200 == 0:
                #percent_done = 100 * batch_idx / len(train_loader)
                #train_losses.append(loss.item())
                #print(f"[{percent_done:>5.1f}%] Batch {batch_idx:>5d} Loss: {loss.item():.4f}")

            if (batch_idx * batch_size) > 1E6: # (Only 1 million samples per epoch when training)
                break

        avg_train_loss = running_train_loss / (batch_idx*batch_size)

        print(f"[TRAIN] Running Saturation Loss: {running_sat_loss/(batch_idx*batch_size):.3e}\tRunning Chem Loss: {running_chem_loss/(batch_idx*batch_size):.3e}")
        print(f"[TRAIN] Running Molar Loss: {running_mole_loss/(batch_idx*batch_size):.3e}\tRunning Bulk Loss: {running_bulk_loss/(batch_idx*batch_size):.3e}")

        # ---- Evaluation ----
        model.eval()
        running_test_loss = 0.0
        running_sat_loss = 0
        running_chem_loss = 0
        running_mole_loss = 0
        running_bulk_loss = 0
        
        with torch.no_grad():
            for batch_idx, (x_batch, b_batch, y_batch, m_batch) in enumerate(test_loader):
                x_batch, b_batch, y_batch, m_batch = x_batch.to(device, non_blocking=True), b_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True), m_batch.to(device, non_blocking=True)
                logits, chem_preds, chem_zero_mask, mole_preds, bulk_preds = model(x_batch, binaries=b_batch, NN_only = True)
                
                # Binary saturation loss
                loss_sat = criterion_sat(logits, b_batch)
                
                # Chemistry losses: apply masks
                chem_loss_raw = criterion_chem(chem_preds, y_batch)
                mole_loss_raw = criterion_mole(mole_preds, m_batch)
                bulk_loss_raw = criterion_bulk(bulk_preds, x_batch[:,3:])
                bulk_zero_mask = (x_batch[:,3:] != 0).to(torch.float)
                #mole_zero_mask = (torch.sigmoid(logits) > 0.5).to(torch.float).detach()
                mole_zero_mask = (b_batch > 0.5).to(torch.float)
                
                chem_loss_masked = (chem_loss_raw * chem_zero_mask*compWeights).sum() / (chem_zero_mask*compWeights).sum().clamp(min=1)
                mole_loss_masked = (mole_loss_raw * mole_zero_mask*binWeights).sum() / (mole_zero_mask*binWeights).sum().clamp(min=1)
                bulk_loss_masked = (bulk_loss_raw * bulk_zero_mask).sum() / bulk_zero_mask.sum().clamp(min=1)
                
                running_sat_loss += loss_sat.item()
                running_mole_loss += mole_loss_masked.item()
                running_chem_loss += chem_loss_masked.item()
                running_bulk_loss += bulk_loss_masked.item()
                
                # Total loss
                #loss = loss_sat + chem_alpha*chem_loss_masked
                #loss = mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
                loss = loss_sat + chem_alpha*chem_loss_masked + mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
                

                running_test_loss += loss.item() * x_batch.size(0)
               


        print(f"[TEST] Running Saturation Loss: {running_sat_loss/len(full_test_set):.3e}\tRunning Chem Loss: {running_chem_loss/len(full_test_set):.3e}")
        print(f"[TEST] Running Molar Loss: {running_mole_loss/len(full_test_set):.3e}\tRunning Bulk Loss: {running_bulk_loss/len(full_test_set):.3e}")

        
        avg_test_loss = running_test_loss / len(full_test_set)
        test_losses.append(avg_test_loss)
        print(f"Epoch {epoch+1:02d}: Train {avg_train_loss:.5f} | Test {avg_test_loss:.5f} | Δt={time.time()-start:.1f}s")

        if avg_test_loss < best_test_loss:
            best_test_loss = avg_test_loss
            torch.save(model.state_dict(), 'Models/temp_upper_train.pt')
            last_missed = False
        elif epoch > 2 and avg_test_loss > best_test_loss * 1.01:
            print("No improvement; stopping early?")
            if last_missed:
                break
            last_missed = True
        

        #ADAPTIVE DROPOUT
        if avg_test_loss > avg_train_loss * 1.02:
            anyDropout = False
            if min(dropout_rate + 0.05, 0.6) > dropout_rate:
                old_drop = dropout_rate
                dropout_rate = min(dropout_rate + 0.05, 0.6)
                for module in model.modules():
                    if isinstance(module, nn.Dropout):
                        module.p = dropout_rate
                        anyDropout  = True
                if anyDropout:
                    print(f"Overfitting. Increasing Dropout: {old_drop} -> {dropout_rate}")
            else: 
                print(f"Overfitting, but dropout_rate is at the maximum: {dropout_rate}")


        elif avg_test_loss < avg_train_loss:
            anyDropout = False
            if max(dropout_rate - 0.02, 0) < dropout_rate:
                old_drop = dropout_rate
                dropout_rate = max(dropout_rate - 0.02, 0) 
                for module in model.modules():
                    if isinstance(module, nn.Dropout):
                        module.p = dropout_rate
                        anyDropout  = True
                if anyDropout:
                    print(f"Underfitting. Decreasing Dropout: {old_drop}->{dropout_rate}")

            else:
                print(f"Underfitting, but dropout_rate is at the minimum: {dropout_rate}")

        gc.collect()

        

    model.load_state_dict(torch.load('Models/temp_upper_train.pt', weights_only=False))
    return model, best_test_loss


def tune_Upper_MELTS(Model, Param_Dict=None, Cr=False, Epochs=7, best_loss = None):
    """
    Function to 
     lower binary saturation model. Initializes new model if none given.
    Best to generate one and give it a description.
    Returns model with best parameters, with the same weights as before.
    """

    import numpy as np
    from copy import deepcopy

    # === Default Param_Dict if none given ===
    if Param_Dict is None:
        Param_Dict = {
            'middleLayer': [[1, 1], [2, 2], [3,3], [1, 0], [2, 1], [3, 2]],
            'high_regularization': ['batchnormdropout0', 'layernormdropout0', 'dropout0'],
            'highWD': [0, 1E-6, 1E-5, 1E-4, 1E-3],
            'noise': [0, 0.01, 0.05, 0.1, 0.2]
        }

        """#Test excluding encoder from adaptive dropout
        if 'dropout' in Model.config['low_regularization'].lower():
            Param_Dict['low_regularization'] = []
            if 'batchnorm' in Model.config['low_regularization'].lower():
                Param_Dict['low_regularization'].append('batchnorm')
            elif 'layernorm' in Model.config['low_regularization'].lower():
                Param_Dict['low_regularization'].append('layernorm')
            else:
                Param_Dict['low_regularization'].append('none')"""

    allowable_keys = ['low_regularization', 'highWD', 'middleLayer', 'high_regularization', 'activation_leak', 'noise']

    for key in list(Param_Dict.keys()):
        assert key in allowable_keys, f"{key} not in {allowable_keys}!"

    # === Baseline training ===
    print("\n" + "=" * 80)
    print(f" BASELINE TRAINING for {Model.config['description']}")
    print("=" * 80)
    if best_loss is None:
        Model, best_loss = train_Upper_MELTS(Model, Cr=Cr, Epochs=Epochs)

    results = [{'model': Model.config, 'loss': best_loss}]
    torch.save({'state_dict': Model.state_dict(), 'config': Model.config}, 'Models/Temp_Upper_Tune.pt')

    # === Begin tuning loop ===
    for parameter, trials in Param_Dict.items():
        print("\n" + "#" * 80)
        print(f" TUNING PARAMETER: {parameter}")
        print("#" * 80)

        # Ensure list type for categorical parameters
        if isinstance(trials, np.ndarray):
            trials = trials.tolist()

        # --- Handle middleBrain Layers (paired parameter) ---
        if parameter == 'middleLayer':
            trials.append([Model.middleLayerUp, Model.middleLayerDown]) # ensure current state represented
            trials = np.array(trials)
            trials = np.unique(trials, axis=0)
            complexity_score = trials[:, 0] - 0.75 * trials[:, 1]
            trials = trials[np.argsort(complexity_score)]
            zero_idx = np.where(
                np.all(trials == np.array([Model.middleLayerUp, Model.middleLayerDown]), axis=1)
            )[0][0]

            current_idx = zero_idx + 1
            go_up = current_idx < trials.shape[0]
            go_down = True

            while 0 <= current_idx < trials.shape[0]:
                
                #working_config = deepcopy(best_config)
                substitutions = {
                    'middleLayerUp': trials[current_idx, 0],
                    'middleLayerDown': trials[current_idx, 1]
                }

                Model = rebuild_MELTS_model('Models/Temp_Upper_Tune.pt', substitutions=substitutions, low_only = True)

                print(Model.config)

                print(f"\nTesting middleLayerUp={substitutions['middleLayerUp']}, "
                      f"middleLayerDown={substitutions['middleLayerDown']}")

                Model, trial_loss = train_Upper_MELTS(Model, Cr=Cr, Epochs=Epochs)
                results.append({'model': Model.config, 'loss': trial_loss})

                if trial_loss < best_loss:
                    print(f"✅ Improved! Loss {trial_loss:.4e} < {best_loss:.4e}")
                    torch.save({'state_dict': Model.state_dict(), 'config': Model.config}, 'Models/Temp_Upper_Tune.pt')

                    best_loss = trial_loss

                    if go_up:
                        go_down = False
                        current_idx += 1
                    else:
                        current_idx -= 1
                elif go_down and go_up:
                    print(f"Going Down... old i: {current_idx}, new i: {zero_idx-1}")
                    current_idx = zero_idx - 1
                    go_up = False
                else:
                    print(f"❌ No improvement — stopping search for {parameter}.")
                    break

            Model = rebuild_MELTS_model('Models/Temp_Upper_Tune.pt') # Rebuild with upper layers too


        else: 
            # Include current parameter value if missing, handled for encoderlayer case above
            current_val = getattr(Model, parameter)
            if current_val is not None and current_val not in trials:
                trials.append(current_val)

        # --- Handle Simple continuous hyperparameters: Weight Decay / noise ---
        if parameter in ['highWD', 'noise']:
            
            changedWD = False # Track if WD/noise changes to load new model at the end. 
            trials = sorted(set(trials))
            zero_idx = trials.index(Model.config[parameter])
            current_idx = zero_idx + 1
            go_up = current_idx < len(trials)
            go_down = True

            while 0 <= current_idx < len(trials):
                
                substitutions = {parameter:trials[current_idx]}

                print(f"\nTesting {parameter}={substitutions[parameter]:.1e}")

                Model = rebuild_MELTS_model('Models/Temp_Upper_Tune.pt', substitutions=substitutions)

                Model, trial_loss = train_Upper_MELTS(Model, Cr=Cr, Epochs=Epochs)
                results.append({'model': Model.config, 'loss': trial_loss})

                if trial_loss < best_loss:
                    print(f"✅ Improved! Loss {trial_loss:.4e} < {best_loss:.4e}")
                    torch.save({'state_dict': Model.state_dict(), 'config': Model.config}, 'Temp_Upper_TuneWD.pt')
                    changedWD = True
                    best_loss = trial_loss
                    if go_up:
                        go_down = False
                        current_idx += 1
                    else:
                        current_idx -= 1
                elif go_down and go_up:
                    current_idx = zero_idx - 1
                    go_up = False
                else:
                    print("❌ No improvement — stopping search for this parameter.")
                    break

            if changedWD:
                Model = rebuild_MELTS_model('Temp_Upper_TuneWD.pt')
                torch.save({'state_dict': Model.state_dict(), 'config': Model.config}, 'Models/Temp_Upper_Tune.pt') # Replace best model with new best
            else: # Rebuild best model
                Model = rebuild_MELTS_model('Models/Temp_Upper_Tune.pt')

        # --- Handle Unordered Categorical Parameters ---
        if parameter in ['activation_leak', 'low_regularization', 'high_regularization']: # Actually activation leak is a continuous parameter and should be treated as such.
            starting_value = getattr(Model, parameter)
            print(f"Debugging: starting categorical parameter redundancy protection: {parameter} = {starting_value}")
            for trial in trials:
                if trial == starting_value:
                    continue

                substitutions = {parameter:trial}
                Model = rebuild_MELTS_model('Models/Temp_Upper_Tune.pt', substitutions=substitutions, low_only=True)

                print(f"\nTesting {parameter}='{trial}'")

                Model, trial_loss = train_Upper_MELTS(Model, Cr=Cr, Epochs=Epochs)
                results.append({'model': Model.config, 'loss': trial_loss})

                if trial_loss < best_loss:
                    print(f"✅ Improved! Loss {trial_loss:.4e} < {best_loss:.4e}")
                    torch.save({'state_dict': Model.state_dict(), 'config': Model.config}, 'Models/Temp_Upper_Tune.pt') 
                    best_loss = trial_loss
                
                else:
                    print(f"❌ No improvement ({trial_loss:.4e})")

            Model = rebuild_MELTS_model('Models/Temp_Upper_Tune.pt')


        # === Summary for this parameter ===
        print("\n" + "-" * 80)
        print(f"→ Current best loss: {best_loss:.4e}")
        print("-" * 80)

    print("\n" + "=" * 80)
    print("TUNING COMPLETE")
    print(f"Best overall loss: {best_loss:.4e}")
    print("=" * 80)

    return Model, results


# AI REVISED:
def tune_Lower_MELTS(Model=None, Param_Dict=None, Cr=False, Epochs=7):
    """
    Function to 
     lower binary saturation model. Initializes new model if none given.
    Best to generate one and give it a description.
    Returns model with best parameters, with the same weights as before.
    """

    import numpy as np
    from copy import deepcopy

    # === Default model if none given ===
    if Model is None:
        Model = NN.MidLevelNetwork(
            encoderLayerUp=1,
            encoderLayerDown=0,
            low_regularization='layernormdropout0',
            description=f"MELTS {MELTSModel}, {CalcType}, {'Cr' if Cr else 'NoCr'}, {date}",
            lowWD = 0#1E-5
        )

    # === Default Param_Dict if none given ===
    if Param_Dict is None:
        Param_Dict = {
            'encoderLayer': [[1, 1], [1, 0], [2, 2], [2, 1], [3, 2]],
            'low_regularization': ['layernormdropout0', 'batchnormdropout0', 'dropout0'],
            'lowWD': [0, 1E-5, 1E-4, 1E-3, 1E-2, 1E-1],
            'noise': [0, 0.01, 0.05, 0.1, 0.2]
        }

    

    allowable_keys = ['low_regularization', 'lowWD', 'encoderLayer', 'activation_leak', 'noise']
    for key in list(Param_Dict.keys()):
        assert key in allowable_keys, f"{key} not in {allowable_keys}!"

    # === Baseline training ===
    print("\n" + "=" * 80)
    print(f" BASELINE TRAINING for {Model.config['description']}")
    print("=" * 80)
    Model, best_loss = train_Lower_MELTS(Model, Cr=Cr, Epochs=Epochs)
    results = [{'model': Model.config, 'loss': best_loss}]
    best_config = deepcopy(Model.config)
    best_weights = Model.state_dict()

    # === Begin tuning loop ===
    for parameter, trials in Param_Dict.items():
        print("\n" + "#" * 80)
        print(f" TUNING PARAMETER: {parameter}")
        print("#" * 80)

        # Ensure list type for categorical parameters
        if isinstance(trials, np.ndarray):
            trials = trials.tolist()

        # --- Handle Encoder Layers (paired parameter) ---
        if parameter == 'encoderLayer':
            trials.append([Model.encoderLayerUp, Model.encoderLayerDown]) # ensure current state represented
            trials = np.array(trials)
            trials = np.unique(trials, axis=0)
            complexity_score = trials[:, 0] - 0.75 * trials[:, 1]
            trials = trials[np.argsort(complexity_score)]
            zero_idx = np.where(
                np.all(trials == np.array([Model.encoderLayerUp, Model.encoderLayerDown]), axis=1)
            )[0][0]

            current_idx = zero_idx + 1
            go_up = current_idx < trials.shape[0]
            go_down = True

            while 0 <= current_idx < trials.shape[0]:
                working_config = deepcopy(best_config)
                working_config['encoderLayerUp'] = trials[current_idx, 0]
                working_config['encoderLayerDown'] = trials[current_idx, 1]

                print(f"\nTesting encoderLayerUp={working_config['encoderLayerUp']}, "
                      f"encoderLayerDown={working_config['encoderLayerDown']}")

                Model = NN.MidLevelNetwork(**working_config)
                Model, trial_loss = train_Lower_MELTS(Model, Cr=Cr, Epochs=Epochs)
                results.append({'model': Model.config, 'loss': trial_loss})

                if trial_loss < best_loss:
                    print(f"✅ Improved! Loss {trial_loss:.4e} < {best_loss:.4e}")
                    best_config = deepcopy(working_config)
                    best_loss = trial_loss
                    best_weights = Model.state_dict()

                    if go_up:
                        go_down = False
                        current_idx += 1
                    else:
                        current_idx -= 1
                elif go_down and go_up:
                    print(f"Going Down... old i: {current_idx}, new i: {zero_idx-1}")
                    current_idx = zero_idx - 1
                    go_up = False
                else:
                    print(f"❌ No improvement — stopping search for {parameter}.")
                    break

            Model = NN.MidLevelNetwork(**best_config)
            Model.load_state_dict(best_weights)

        else: 
            # Include current parameter value if missing, handled for encoderlayer case above
            current_val = getattr(Model, parameter)
            if current_val is not None and current_val not in trials:
                trials.append(current_val)

        # --- Handle Simple Continuous Hyperparameters: Weight Decay, noise ---
        if parameter in ['lowWD', 'noise']:
            best_weights_WD = deepcopy(best_weights)
            trials = sorted(set(trials))
            zero_idx = trials.index(Model.config[parameter])
            current_idx = zero_idx + 1
            go_up = current_idx < len(trials)
            go_down = True

            while 0 <= current_idx < len(trials):
                working_config = deepcopy(best_config)
                working_config[parameter] = trials[current_idx]

                print(f"\nTesting {parameter}={working_config[parameter]:.1e}")

                Model = NN.MidLevelNetwork(**working_config)
                Model.load_state_dict(best_weights) # Since WD is not model structure, load in best weights)
                Model, trial_loss = train_Lower_MELTS(Model, Cr=Cr, Epochs=Epochs)
                results.append({'model': Model.config, 'loss': trial_loss})

                if trial_loss < best_loss:
                    print(f"✅ Improved! Loss {trial_loss:.4e} < {best_loss:.4e}")
                    best_config = deepcopy(working_config)
                    best_loss = trial_loss
                    best_weights_WD = Model.state_dict() # Save state dict without loading it for the next WD trial for fairness
                    if go_up:
                        go_down = False
                        current_idx += 1
                    else:
                        current_idx -= 1
                elif go_down and go_up:
                    current_idx = zero_idx - 1
                    go_up = False
                else:
                    print("❌ No improvement — stopping search for this parameter.")
                    break

            Model = NN.MidLevelNetwork(**best_config)
            best_weights = best_weights_WD
            

        # --- Handle Unordered Categorical Parameters: Try every option with no early abort ---
        if parameter in ['activation_factory', 'low_regularization']:
            starting_value = getattr(Model, parameter)
            print(f"Debugging: starting categorical parameter redundancy protection: {parameter} = {starting_value}")
            for trial in trials:
                if trial == starting_value:
                    continue

                working_config = deepcopy(best_config)
                working_config[parameter] = trial

                print(f"\nTesting {parameter}='{trial}'")

                Model = NN.MidLevelNetwork(**working_config)
                Model, trial_loss = train_Lower_MELTS(Model, Cr=Cr, Epochs=Epochs)
                results.append({'model': Model.config, 'loss': trial_loss})

                if trial_loss < best_loss:
                    print(f"✅ Improved! Loss {trial_loss:.4e} < {best_loss:.4e}")
                    best_config = deepcopy(working_config)
                    best_loss = trial_loss
                    best_weights = Model.state_dict()

                else:
                    print(f"❌ No improvement ({trial_loss:.4e})")

            Model = NN.MidLevelNetwork(**best_config)

        # === Summary for this parameter ===
        print("\n" + "-" * 80)
        print(f"Best {parameter} configuration so far:")
        for k, v in best_config.items():
            print(f"  {k}: {v}")
        print(f"→ Current best loss: {best_loss:.4e}")
        print("-" * 80)

    print("\n" + "=" * 80)
    print("TUNING COMPLETE")
    print(f"Best overall loss: {best_loss:.4e}")
    print("=" * 80)

    Model.load_state_dict(best_weights) # Load best model's weights

    return Model, results


In [ ]:
"""Binary Phase Saturation Model With Adaptive Dropout"""
from BackEnds.nnMELTS import rebuild_MELTS_model


modelname =f"MELTS{MELTSModel}{CalcType}"

#NoCrModel, CrModel = models['NoCr'], models['Cr']

criterion = nn.BCEWithLogitsLoss()  # suitable for multi-label classification
#criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights.cuda())  # suitable for multi-label classification, weighting rare phases
device = 'cuda'

#Reg = 'layernormdropout0'
#wd = 0.1
#noise = 0.001
date = 'Nov13'
batch_size = 1024


#Get Best Models
NoCrModel, NoCrresults = tune_Lower_MELTS(Model=None, Param_Dict=None, Cr = False, Epochs = 7)
CrModel, Crresults = tune_Lower_MELTS(Model=NoCrModel, Param_Dict=None, Cr = True, Epochs = 7)
CrModel.config['description'] = CrModel.config['description'].replace('No','') # Make it clear that this is YES chromium.
CrModel.description = CrModel.config['description']

"""DictFilePath=f"Models/MELTS{MELTSModel}{CalcType}Cr_BinaryOnly_{date}.pt"
NoCrModel = rebuild_MELTS_model(DictFilePath)
CrModel = rebuild_MELTS_model(DictFilePath) """

for i, (binary_train_set, binary_test_set, FullMELTS) in enumerate([(binary_train_set_NoCr, binary_test_set_NoCr, NoCrModel), (binary_train_set_Cr, binary_test_set_Cr, CrModel)]):

    #if i == 0:
    #    continue
    print(['NoCr', 'Cr'][i])
    print(FullMELTS)
    
    
    #if i == 0:
        #continue
    
    FullMELTS.cuda()

    #modelname = "rhyoliteMELTS1.1.0Batch"
    DictFilePath=f"Models/MELTS{MELTSModel}{CalcType}{['NoCr', 'Cr'][i]}_BinaryOnly_{date}.pt"
    #FullMELTS.load_state_dict(torch.load(DictFilePath), strict=False) #Warm Start
    #ictFilePath=f'./{modelname}_BinaryOnly0.0025noise_{date}.pt'
    #date = "Oct11"
    #DictFilePath=f"Models/MELTS{MELTSModel}{CalcType}{['NoCr','Cr'][i]}_BinaryOnly_{date}.pt"
    
    Reg = FullMELTS.config['low_regularization']    
    wd = FullMELTS.config['lowWD']
    noise = FullMELTS.config['noise']

    if 'dropout' in Reg.lower():
        dropout_rate = pull_number(Reg)
    else:
        dropout_rate = 0
    binary_train_loader = DataLoader(binary_train_set, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    binary_test_loader = DataLoader(binary_test_set, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    
    for p in FullMELTS.parameters():
        p.requires_grad = True
    for p in FullMELTS.chem_heads.parameters():
        p.requires_grad = False
    for p in FullMELTS.mole_head.parameters():
        p.requires_grad = False
    if FullMELTS.middleBrain is not None:
        for p in FullMELTS.middle_brain.parameters():
            p.requires_grad = False

    train_losses = []
    test_losses = []
    long_test_losses = []
    valid_loss_min_binary = np.inf

    #EPOCHS = 50
    epoch = 0
    improve_record = [1,1,1,1]
    #lrs = np.logspace(-8,-4,9).tolist() 
    lrs = np.logspace(-7,-3,9).tolist() 

    lr = lrs.pop()
    optimizer = AdamW(FullMELTS.parameters(), lr=lr, weight_decay=wd)


    # ---- Training loop ----
    #for epoch in range(EPOCHS):
    while lr > 2E-7:

        epoch += 1
        epoch_start = time.time()
        FullMELTS.train()
        running_train_loss = 0.0
        total_batches = len(binary_train_loader)

        print(f"\n--- Epoch {epoch} ---") #/{EPOCHS}

        for batch_idx, (x_batch, y_batch) in enumerate(tqdm(binary_train_loader, desc="Training", leave=False)):
            x_batch, y_batch = x_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True)
            if noise != 0:
                x_batch = x_batch + (x_batch * torch.randn_like(x_batch) * noise)
            #x_batch = x_batch + torch.randn_like(x_batch) * noise # Add Small Gaussian Noise to avoid overfitting during training

            optimizer.zero_grad()
            logits = FullMELTS.forward_binaries(x_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

            running_train_loss += loss.item() * x_batch.size(0)

            if batch_idx % 200 == 0:
                percent_done = 100 * batch_idx / total_batches
                train_losses.append(loss.item())
                print(f"[{percent_done:>5.1f}%] Batch {batch_idx:>5d} Loss: {loss.item():.4f}")

        avg_train_loss = running_train_loss / len(binary_train_set)


        # ---- Evaluation ----
        FullMELTS.eval()
        running_test_loss = 0.0
        with torch.no_grad():
            for batch_idx, (x_batch, y_batch) in enumerate(binary_test_loader):
                x_batch, y_batch = x_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True)
                logits = FullMELTS.forward_binaries(x_batch)
                loss = criterion(logits, y_batch)
                running_test_loss += loss.item() * x_batch.size(0)
                if batch_idx % 50 == 0:
                    long_test_losses.append(loss.item())

        avg_test_loss = running_test_loss / len(binary_test_set)
        test_losses.append(avg_test_loss)
        print(f"Running Saturation Loss: {round(running_test_loss,4)}")
        if avg_test_loss <= valid_loss_min_binary:
            #torch.save(FullMELTS.state_dict(), DictFilePath)
            #torch.save({'state_dict': FullMELTS.state_dict(), 'config': FullMELTS.config}, DictFilePath)
            FullMELTS.save(DictFilePath)
            print('\tValidation loss decreased ({:.6f} --> {:.6f}).  Saving model ...'.format(valid_loss_min_binary, avg_test_loss))
            valid_loss_min_binary = avg_test_loss
            improve_record.append(1)
        else:
            improve_record.append(0)

        """if avg_train_loss<(avg_test_loss*0.95):
            print(f"Overfitting! Increasing Gaussian Noise: {noise}->{noise+0.001}")
            noise += 0.001"""

        print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.6f} | Test Loss: {avg_test_loss:.6f}")#/{EPOCHS}
        print(f"[TIMER] Epoch time: {time.time() - epoch_start:.2f} seconds")
        if np.sum(improve_record[-4:]) == 0:
            improve_record = [1,1,1,1]
            lr = lrs.pop()
            """if wd > 5*lr:
                wd = 5*lr"""
            print(f"No Improvement in 3 epochs. New LR: {lr}. New Weight Decay: {wd}")
            optimizer = AdamW(FullMELTS.parameters(), lr=lr, weight_decay=wd)

        #ADAPTIVE DROPOUT
        if avg_test_loss > avg_train_loss * 1.02:
            anyDropout = False
            if min(dropout_rate + 0.05, 0.6) > dropout_rate:
                old_drop = dropout_rate
                dropout_rate = min(dropout_rate + 0.05, 0.6)
                for module in FullMELTS.modules():
                    if isinstance(module, nn.Dropout):
                        module.p = dropout_rate
                        anyDropout  = True
                if anyDropout:
                    print(f"Overfitting. Increasing Dropout: {old_drop} -> {dropout_rate}")
            else: 
                print(f"Overfitting, but dropout_rate is at the maximum: {dropout_rate}")


        elif avg_test_loss < avg_train_loss:
            anyDropout = False
            if max(dropout_rate - 0.02, 0) < dropout_rate:
                old_drop = dropout_rate
                dropout_rate = max(dropout_rate - 0.02, 0) 
                for module in FullMELTS.modules():
                    if isinstance(module, nn.Dropout):
                        module.p = dropout_rate
                        anyDropout  = True
                if anyDropout:
                    print(f"Underfitting. Decreasing Dropout: {old_drop}->{dropout_rate}")

            else:
                print(f"Underfitting, but dropout_rate is at the minimum: {dropout_rate}")

                
    # ---- Plotting ----
    plt.figure(figsize=(8, 5))
    plt.plot((epoch/len(train_losses))*(np.arange(len(train_losses))+1),train_losses, label='Train Loss')
    plt.plot((epoch/len(long_test_losses))*(np.arange(len(long_test_losses))+1), long_test_losses, label='Test Loss')
    plt.xlabel("Epoch")
    plt.ylabel("Loss (BCEWithLogits)")
    plt.title("Phase Saturation Training and Test Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"{DictFilePath.split('.')[0]}_BinaryPhaseSatTrain_{date}.jpg", dpi = 256)
    plt.show()

    """Histograms of Binary Phase Saturation Probabilities"""

    directories = [f"{DictFilePath.split('.')[0]}Binary_Phase_Saturation_Histograms_TRAIN",f"{DictFilePath.split('.')[0]}Binary_Phase_Saturation_Histograms_TEST"]
   
    for i, histogram_directory in enumerate(directories):

        if not os.path.exists(histogram_directory):
            os.makedirs(histogram_directory)
        if len([binary_train_set, binary_test_set][i]) > 500000:
            subset = np.random.choice(np.arange(0, len([binary_train_set, binary_test_set][i])), size=500000, replace=False)
        else:
            subset = np.arange(0, len([binary_train_set, binary_test_set][i]))

        Xtest, Ytest = ([binary_train_set, binary_test_set][i])[subset.tolist()]
        with torch.no_grad():
            Y_hat_test = torch.sigmoid(FullMELTS.forward_binaries(Xtest.to('cuda')))
            Y_hat_test = Y_hat_test.detach().cpu().numpy()
            Xtest = Xtest.detach().numpy()
            Ytest = Ytest.detach().numpy()
        gc.collect()
        with open(histogram_directory+'/PRstats.txt', 'w'): # Create blank file
            pass


        for i, phase in enumerate(list(label_indices.keys())):
            realPos = (Ytest[:,i] > 0.5)
            predPos = (Y_hat_test[:,i] > 0.5).astype(float)
            precision = Ytest[predPos.astype(bool),i].sum()/np.sum(predPos)
            recall = Y_hat_test[realPos.astype(bool),i].sum()/np.sum(realPos)
            with open(histogram_directory+'/PRstats.txt', 'a') as File: #Record Stats
                File.write(f"Model {phase} Precision of positive prediction: {round(100*precision,2)}\n")
                File.write(f"Model {phase} Recall of dataset positives : {round(100*recall,2)}%\n")
            print(f"Model {phase} Precision of positive prediction: {round(100*precision,2)}% ")
            print(f"Model {phase} Recall of dataset positives : {round(100*recall,2)}%")
            plt.hist(Y_hat_test[realPos.astype(bool),i], bins=30, alpha=0.5, color = 'blue', label=f'{phase} Present', density=True, log = True)
            plt.hist(Y_hat_test[~(realPos.astype(bool)),i], bins=30, alpha=0.5, color = 'red', label=f'{phase} Absent', density=True, log = True)
            #plt.axvline(x=3, color='r', linestyle='dashed', linewidth=1)
            plt.legend()
            plt.xlabel("Probability")
            plt.ylabel("Normalized Frequency (Log Scale)")
            plt.title(f"NN {phase} Saturation Probabilities\nPresent in {round(100*realPos.sum()/len(Ytest),2)}% of Dataset")
            plt.tight_layout()
            plt.savefig(histogram_directory+f"/{phase}_Saturation_Probability_Histogram")
            plt.show()

In [ ]:
print(Crresults)
print(NoCrresults)

In [ ]:
#Chem and Mole head training. (WEIGHTS DEACTIVATED) NoBulk, frozen encodings
#torch.autograd.set_detect_anomaly(False)
#

from BackEnds.nnMELTS import *

importlib.reload(NN)

substitutions = {
    'middleLayerUp':2,
    'middleLayerDown':1,
    'high_regularization':'dropout0',
    'highWD':1E-5
}

tune = True # Did False 11/3/25
if tune:
    for i, suffix in enumerate(['NoCr', 'Cr']):
        #if i == 0: 
        #    continue
        DictFilePath=f"Models/MELTS{MELTSModel}{CalcType}{suffix}_BinaryOnly_{date}.pt"
        Model = rebuild_MELTS_model(DictFilePath, substitutions=substitutions, low_only = True)
        if bool(i): 
            CrModel, Crresults = tune_Upper_MELTS(Model, Param_Dict=None, Cr=bool(i), Epochs=7)
            CrModel.save(f"Models/MELTS{MELTSModel}{CalcType}{suffix}_Tuned_{date}.pt")
            print(CrModel.config)
        else:
            NoCrModel, NoCrresults = tune_Upper_MELTS(Model, Param_Dict=None, Cr=bool(i), Epochs=7)
            NoCrModel.save(f"Models/MELTS{MELTSModel}{CalcType}{suffix}_Tuned_{date}.pt")
            print(NoCrModel.config)
else:
    CrModel, NoCrModel = rebuild_MELTS_model(f"Models/MELTS{MELTSModel}{CalcType}Cr_Tuned_{date}.pt"), rebuild_MELTS_model(f"Models/MELTS{MELTSModel}{CalcType}NoCr_Tuned_{date}.pt")

#CHANGE DATA FOR REMAIDER OF TRAINING
###############################
#date = 'Nov3(NoNoise)'
###############################


weight_dict_both = {
    'olivine': 3,
    'nepheline': 15,
    'leucite': 10,
    'alloy-solid': 5,
    'muscovite': 3,
    'k-feldspar': 5
}
# This overprints the above weights, does not apply to NoCr model
weight_dict_Cr = {
    'rhm-oxide': 5
}

binWeightsNoCr = torch.ones(len(list(label_indices.keys())), dtype = torch.float32).reshape(1,-1)
binWeightsCr = torch.ones(len(list(label_indices.keys())), dtype = torch.float32).reshape(1,-1)
compWeightsNoCr = torch.ones(label_indices_comp['melts-liquid'][-1]+1, dtype = torch.float32).reshape(1,-1)
compWeightsCr = torch.ones(label_indices_comp['melts-liquid'][-1]+1, dtype = torch.float32).reshape(1,-1)
"""for phase, W in weight_dict_both.items(): # Weights removed for 110 due to bad overfitting... Take a look at phase abundances??? 
    binWeightsCr[:,mass_phasedict[phase]] = W
    binWeightsNoCr[:,mass_phasedict[phase]] = W
    if phase in compositionally_variable_phases:
        compWeightsCr[:,comp_phasedict[phase]] = W
        compWeightsNoCr[:,comp_phasedict[phase]] = W
for phase, W in weight_dict_Cr.items():
    binWeightsCr[:,mass_phasedict[phase]] = W
    if phase in compositionally_variable_phases:
        compWeightsCr[:,comp_phasedict[phase]] = W"""


for i, (full_train_set, full_test_set) in enumerate([(full_train_set_NoCr, full_test_set_NoCr), (full_train_set_Cr, full_test_set_Cr)]):
    #if i ==1:
    #    continue
    FullMELTS = [NoCrModel, CrModel][i]
    #DualSaturationChemistry().cuda()
    #date = "Sept30"
    #modelname = "rhyoliteMELTS1.0.2Fxtal"
    ##DictFilePath=f'./{modelname}_ChemMoleL2_{date}.pt'
    #date = "Oct16"
    #DictFilePath=f"Models/MELTS{MELTSModel}{CalcType}{['NoCr','Cr'][i]}_BinaryOnlyLayerNorm_{date}.pt"
    #DictFilePath=f"Models/{modelname}{['NoCr', 'Cr'][i]}_ChemMoleL2_{date}.pt"

    #DictFilePath=f"Models/{modelname}{['NoCr', 'Cr'][i]}_BinaryOnly_{date}.pt"
    #FullMELTS.load_state_dict(torch.load(DictFilePath),strict = False)
    

    #date = "Sept22" + ['NoCr', 'Cr'][i]
    #date = 'Oct11'
    #DictFilePath=f"Models/{modelname}{['NoCr', 'Cr'][i]}_ChemMoleL2_{date}.pt"
    DictFilePath = f"Models/MELTS{MELTSModel}{CalcType}{['NoCr','Cr'][i]}_ChemMoleL2_{date}.pt"
    
    Reg = FullMELTS.config['high_regularization']    
    wd = FullMELTS.config['highWD']
    noise = FullMELTS.config['noise']

    if 'dropout' in Reg.lower():
        dropout_rate = pull_number(Reg)
    else:
        dropout_rate = 0
    
    batch_size = 1024

    device = 'cuda'

    binWeights = [binWeightsNoCr, binWeightsCr][i]
    binWeights = binWeights.to(device)
    compWeights = [compWeightsNoCr, compWeightsCr][i]
    compWeights = (compWeights).to(device)
    FullMELTS = FullMELTS.cuda()
    
    train_loader = DataLoader(full_train_set, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    test_loader = DataLoader(full_test_set, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    # All but mole heads: Put those on top.
    for p in FullMELTS.parameters():
        p.requires_grad = True
    #for p in FullMELTS.mole_head.parameters():
    #    p.requires_grad = False
    for p in FullMELTS.encoder.parameters():
        p.requires_grad = False
    for p in FullMELTS.sat_head.parameters():
        p.requires_grad = False
        
        
    criterion_sat = torch.nn.BCEWithLogitsLoss(weight = binWeights)
    criterion_chem = symmetric_rel_l2 #relative_L1_loss #y_pred, y_true, mask=None, eps=1e-6 #criterion_chem = F.mse_loss 
    criterion_mole = symmetric_rel_l2
    criterion_bulk = symmetric_rel_l2
    
    chem_alpha = 1#200 # Oct 16: moved from 1->100 due to two order of magnitude difference between saturation and other losses 
    mole_alpha = 1#20
    bulk_alpha = 0 # For now, guassian noise in bulk
    

    train_losses = []
    test_losses = []
    long_test_losses = []
    valid_loss_min = np.inf


    epoch = 0
    improve_record = [1,1,1,1]
    lrs = np.logspace(-7,-3,9).tolist()
    lr = lrs.pop()
    #wd = 1E-4 # Oct 16: moved 1E-5 -> 1E-4 was 1E-5 for MELTS102 before forcing superliquidus to regurgitate inputs 
    optimizer = AdamW(filter(lambda p: p.requires_grad, FullMELTS.parameters()), lr=lr, weight_decay=wd)



    # ---- Training loop ----
    #for epoch in range(EPOCHS):
    while (lr > 2E-7):
    #while lr > 2E-25:

        epoch += 1
        epoch_start = time.time()
        FullMELTS.train()
        running_train_loss = 0.0
        total_batches = len(train_loader)

        running_sat_loss = 0
        running_chem_loss = 0
        running_mole_loss = 0
        running_bulk_loss = 0

        print(f"\n--- Epoch {epoch} ---") #/{EPOCHS}

        for batch_idx, (x_batch, b_batch, y_batch, m_batch) in enumerate(tqdm(train_loader, desc="Training", leave=False)):

            optimizer.zero_grad()

            x_batch, b_batch, y_batch, m_batch = x_batch.to(device, non_blocking=True), b_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True), m_batch.to(device, non_blocking=True)
            
            bulk_zero_mask = (x_batch != 0).to(torch.float) # Bulk zero mask different shape for training and testing because this mask is doubling as a filter for the noise
            #x_batch = x_batch + torch.randn_like(x_batch) * 0.005 * bulk_zero_mask # Add Small Gaussian Noise to avoid overfitting during training Oct 14: Moved 0.002->0.005
            if noise != 0:
                x_batch = x_batch + (x_batch * torch.randn_like(x_batch) * noise)
                
            logits, chem_preds, chem_zero_mask, mole_preds, bulk_preds = FullMELTS(x_batch, binaries=b_batch, NN_only = True)

            # Binary saturation loss
            loss_sat = criterion_sat(logits, b_batch) 

            # Chemistry and mass losses: apply masks
            chem_loss_raw = criterion_chem(chem_preds, y_batch)
            mole_loss_raw = criterion_mole(mole_preds, m_batch)
            bulk_loss_raw = criterion_bulk(bulk_preds, x_batch[:,3:])
            #mole_zero_mask = (torch.sigmoid(logits) > 0.5).to(torch.float).detach() # Only use binary preds for masking when those neurons are free
            mole_zero_mask = (b_batch > 0.5).to(torch.float).detach()
     
            chem_loss_masked = (chem_loss_raw * chem_zero_mask * compWeights).sum() / (chem_zero_mask * compWeights).sum().clamp(min=1)
            mole_loss_masked = (mole_loss_raw * mole_zero_mask * binWeights).sum() / (mole_zero_mask * binWeights).sum().clamp(min=1)
            bulk_loss_masked = (bulk_loss_raw * bulk_zero_mask[:,3:]).sum() / (bulk_zero_mask[:,3:]).sum().clamp(min=1)
            
            running_sat_loss += loss_sat.item()
            running_mole_loss += mole_loss_masked.item()
            running_chem_loss += chem_loss_masked.item()
            running_bulk_loss += bulk_loss_masked.item()
            
            
            
            loss = loss_sat + chem_alpha*chem_loss_masked + mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
            #print(f"Sat loss: {loss_sat.item()}, Chem loss: {chem_loss_masked.item()}")

            if not torch.isfinite(loss):
                print("Non-finite loss detected!")
                print(f"Sat loss: {loss_sat.item()}, Chem loss: {chem_loss_masked.item()}, Mole loss: {mole_loss_masked}, Bulk loss: {bulk_loss_masked}")
                continue

            loss.backward()


            optimizer.step()

            running_train_loss += loss.item() * x_batch.size(0)

            """if batch_idx % 200 == 0:
                percent_done = 100 * batch_idx / total_batches
                train_losses.append(loss.item())
                print(f"[{percent_done:>5.1f}%] Batch {batch_idx:>5d} Loss: {loss.item():.4f}")"""


        avg_train_loss = running_train_loss / len(full_train_set)

        print(f"Running Saturation Loss: {running_sat_loss/len(full_train_set):.3e}\nRunning Chem Loss: {running_chem_loss/len(full_train_set):.3e}")
        print(f"Running Molar Loss: {running_mole_loss/len(full_train_set):.3e}\nRunning Bulk Loss: {running_bulk_loss/len(full_train_set):.3e}")

        # ---- Evaluation ----
        FullMELTS.eval()
        running_test_loss = 0.0
        running_sat_loss = 0
        running_chem_loss = 0
        running_mole_loss = 0
        running_bulk_loss = 0
        
        with torch.no_grad():
            for batch_idx, (x_batch, b_batch, y_batch, m_batch) in enumerate(test_loader):
                x_batch, b_batch, y_batch, m_batch = x_batch.to(device, non_blocking=True), b_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True), m_batch.to(device, non_blocking=True)
                logits, chem_preds, chem_zero_mask, mole_preds, bulk_preds = FullMELTS(x_batch, binaries=b_batch, NN_only = True)
                
                # Binary saturation loss
                loss_sat = criterion_sat(logits, b_batch)
                
                # Chemistry losses: apply masks
                chem_loss_raw = criterion_chem(chem_preds, y_batch)
                mole_loss_raw = criterion_mole(mole_preds, m_batch)
                bulk_loss_raw = criterion_bulk(bulk_preds, x_batch[:,3:])
                bulk_zero_mask = (x_batch[:,3:] != 0).to(torch.float)
                #mole_zero_mask = (torch.sigmoid(logits) > 0.5).to(torch.float).detach()
                mole_zero_mask = (b_batch > 0.5).to(torch.float)
                
                chem_loss_masked = (chem_loss_raw * chem_zero_mask*compWeights).sum() / (chem_zero_mask*compWeights).sum().clamp(min=1)
                mole_loss_masked = (mole_loss_raw * mole_zero_mask*binWeights).sum() / (mole_zero_mask*binWeights).sum().clamp(min=1)
                bulk_loss_masked = (bulk_loss_raw * bulk_zero_mask).sum() / bulk_zero_mask.sum().clamp(min=1)
                
                running_sat_loss += loss_sat.item()
                running_mole_loss += mole_loss_masked.item()
                running_chem_loss += chem_loss_masked.item()
                running_bulk_loss += bulk_loss_masked.item()
                
                # Total loss
                #loss = loss_sat + chem_alpha*chem_loss_masked
                #loss = mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
                loss = loss_sat + chem_alpha*chem_loss_masked + mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
                

                running_test_loss += loss.item() * x_batch.size(0)
                if batch_idx % 50 == 0:
                    long_test_losses.append(loss.item())


        print(f"Running Saturation Loss: {running_sat_loss/len(full_test_set):.3e}\nRunning Chem Loss: {running_chem_loss/len(full_test_set):.3e}")
        print(f"Running Molar Loss: {running_mole_loss/len(full_test_set):.3e}\nRunning Bulk Loss: {running_bulk_loss/len(full_test_set):.3e}")

        
        avg_test_loss = running_test_loss / len(full_test_set)
        test_losses.append(avg_test_loss)
        if avg_test_loss <= valid_loss_min:
            #torch.save(FullMELTS.state_dict(), DictFilePath)
            FullMELTS.save(DictFilePath)
            print('\tValidation loss decreased ({:.6f} --> {:.6f}).  Saving model ...'.format(valid_loss_min, avg_test_loss))
            valid_loss_min = avg_test_loss
            improve_record.append(1)
        else:
            improve_record.append(0)

        print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.6f} | Test Loss: {avg_test_loss:.6f}")#/{EPOCHS}
        print(f"[TIMER] Epoch time: {time.time() - epoch_start:.2f} seconds")
        if np.sum(improve_record[-4:]) == 0:
            improve_record = [1,1,1,1]
            lr = lrs.pop()
            #if wd > 5*lr:
            #    wd = 5*lr
            print(f"No Improvement in 4 epochs. New LR: {lr}. New Weight Decay: {wd}")
            optimizer = AdamW(filter(lambda p: p.requires_grad, FullMELTS.parameters()), lr=lr, weight_decay=wd)#Adam(FullMELTS.parameters(), lr=lr)#, weight_decay=wd)
        #Adam(FullMELTS.parameters(), lr=lr)#, weight_decay=wd)

        #ADAPTIVE DROPOUT
        if avg_test_loss > avg_train_loss * 1.02:
            anyDropout = False
            if min(dropout_rate + 0.05, 0.6) > dropout_rate:
                old_drop = dropout_rate
                dropout_rate = min(dropout_rate + 0.05, 0.6)
                for module in FullMELTS.modules():
                    if isinstance(module, nn.Dropout):
                        module.p = dropout_rate
                        anyDropout  = True
                if anyDropout:
                    print(f"Overfitting. Increasing Dropout: {old_drop} -> {dropout_rate}")
            else: 
                print(f"Overfitting, but dropout_rate is at the maximum: {dropout_rate}")


        elif avg_test_loss < avg_train_loss:
            anyDropout = False
            if max(dropout_rate - 0.02, 0) < dropout_rate:
                old_drop = dropout_rate
                dropout_rate = max(dropout_rate - 0.02, 0) 
                for module in FullMELTS.modules():
                    if isinstance(module, nn.Dropout):
                        module.p = dropout_rate
                        anyDropout  = True
                if anyDropout:
                    print(f"Underfitting. Decreasing Dropout: {old_drop}->{dropout_rate}")

            else:
                print(f"Underfitting, but dropout_rate is at the minimum: {dropout_rate}")

        
        #break

In [ ]:
chem_loss_raw

In [ ]:
print(Crresults)
print(NoCrresults)

In [ ]:
#FullL1. 
#torch.autograd.set_detect_anomaly(False)

#date = 'Nov5'

weight_dict_both = {
    'olivine': 3,
    'nepheline': 15,
    'leucite': 10,
    'alloy-solid': 5,
    'muscovite': 3,
    'k-feldspar': 5
}
# This overprints the above weights, does not apply to NoCr model
weight_dict_Cr = {
    'rhm-oxide': 5
}

binWeightsNoCr = torch.ones(len(list(label_indices.keys())), dtype = torch.float32).reshape(1,-1)
binWeightsCr = torch.ones(len(list(label_indices.keys())), dtype = torch.float32).reshape(1,-1)
compWeightsNoCr = torch.ones(label_indices_comp['melts-liquid'][-1]+1, dtype = torch.float32).reshape(1,-1)
compWeightsCr = torch.ones(label_indices_comp['melts-liquid'][-1]+1, dtype = torch.float32).reshape(1,-1)
"""for phase, W in weight_dict_both.items(): # Weights removed for 110 due to bad overfitting... Take a look at phase abundances??? 
    binWeightsCr[:,mass_phasedict[phase]] = W
    binWeightsNoCr[:,mass_phasedict[phase]] = W
    if phase in compositionally_variable_phases:
        compWeightsCr[:,comp_phasedict[phase]] = W
        compWeightsNoCr[:,comp_phasedict[phase]] = W
for phase, W in weight_dict_Cr.items():
    binWeightsCr[:,mass_phasedict[phase]] = W
    if phase in compositionally_variable_phases:
        compWeightsCr[:,comp_phasedict[phase]] = W"""


for i, (full_train_set, full_test_set) in enumerate([(full_train_set_NoCr, full_test_set_NoCr), (full_train_set_Cr, full_test_set_Cr)]):

    #FullMELTS = [NoCrModel, CrModel][i]
    
    #DualSaturationChemistry().cuda()
    #date = "Sept30"
    #modelname = "rhyoliteMELTS1.0.2Fxtal"
    ##DictFilePath=f'./{modelname}_ChemMoleL2_{date}.pt'
    #date = "Oct16"
    date = 'Nov13'
    DictFilePath=f"Models/MELTS{MELTSModel}{CalcType}{['NoCr','Cr'][i]}_ChemMoleL2_{date}.pt"
    DictFilePath=f"Models/{modelname}{['NoCr', 'Cr'][i]}_FullL1_{date}.pt"

    FullMELTS = rebuild_MELTS_model(DictFilePath)
    date = 'Nov17'

    #DictFilePath=f"Models/{modelname}{['NoCr', 'Cr'][i]}_BinaryOnly_{date}.pt"
    #FullMELTS.load_state_dict(torch.load(DictFilePath),strict = False)
    

    #date = "Sept22" + ['NoCr', 'Cr'][i]
    #date = 'Oct11'
    DictFilePath=f"Models/{modelname}{['NoCr', 'Cr'][i]}_FullL1_{date}.pt"
    
    Reg = FullMELTS.config['high_regularization']    
    wd = FullMELTS.config['highWD']
    noise = FullMELTS.config['noise']

    if 'dropout' in Reg.lower():
        dropout_rate = pull_number(Reg)
    else:
        dropout_rate = 0
    
    
    batch_size = 1024

    device = 'cuda'

    binWeights = [binWeightsNoCr, binWeightsCr][i]
    binWeights = binWeights.to(device)
    compWeights = [compWeightsNoCr, compWeightsCr][i]
    compWeights = (compWeights).to(device)
    FullMELTS = FullMELTS.cuda()
    
    train_loader = DataLoader(full_train_set, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    test_loader = DataLoader(full_test_set, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    # All 
    for p in FullMELTS.parameters():
        p.requires_grad = True
    #for p in FullMELTS.mole_head.parameters():
    #    p.requires_grad = False
    """for p in FullMELTS.encoder.parameters():
        p.requires_grad = False
    for p in FullMELTS.sat_head.parameters():
        p.requires_grad = False"""
        
        
    criterion_sat = torch.nn.BCEWithLogitsLoss(weight = binWeights)
    criterion_chem = symmetric_rel_l1 #relative_L1_loss #y_pred, y_true, mask=None, eps=1e-6 #criterion_chem = F.mse_loss 
    criterion_mole = symmetric_rel_l1
    criterion_bulk = symmetric_rel_l1
    
    chem_alpha = 1#100#200 # Oct 16: moved from 1->100 due to two order of magnitude difference between saturation and other losses 
    mole_alpha = 1#100#20
    bulk_alpha = 1#50#20
    

    train_losses = []
    test_losses = []
    long_test_losses = []
    valid_loss_min = np.inf


    epoch = 0
    improve_record = [1,1,1,1]
    lrs = np.logspace(-7,-3,9).tolist()
    lr = lrs.pop()
    wd = FullMELTS.highWD # Oct 16: moved 1E-5 -> 1E-4 was 1E-5 for MELTS102 before forcing superliquidus to regurgitate inputs 
    optimizer = AdamW(filter(lambda p: p.requires_grad, FullMELTS.parameters()), lr=lr, weight_decay=wd)

    # ---- Training loop ----
    #for epoch in range(EPOCHS):
    while (lr > 2E-7):
    #while lr > 2E-25:

        epoch += 1
        epoch_start = time.time()
        FullMELTS.train()
        running_train_loss = 0.0
        total_batches = len(train_loader)

        running_sat_loss = 0
        running_chem_loss = 0
        running_mole_loss = 0
        running_bulk_loss = 0

        print(f"\n--- Epoch {epoch} ---") #/{EPOCHS}

        for batch_idx, (x_batch, b_batch, y_batch, m_batch) in enumerate(tqdm(train_loader, desc="Training", leave=False)):

            optimizer.zero_grad()

            x_batch, b_batch, y_batch, m_batch = x_batch.to(device, non_blocking=True), b_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True), m_batch.to(device, non_blocking=True)
            
            bulk_zero_mask = (x_batch != 0).to(torch.float) # Bulk zero mask different shape for training and testing because this mask is doubling as a filter for the noise
            #x_batch = x_batch + torch.randn_like(x_batch) * 0.005 * bulk_zero_mask # Add Small Gaussian Noise to avoid overfitting during training Oct 14: Moved 0.002->0.005

            logits, chem_preds, chem_zero_mask, mole_preds, bulk_preds = FullMELTS(x_batch, binaries=b_batch, NN_only = True)

            # Binary saturation loss
            loss_sat = criterion_sat(logits, b_batch) 

            # Chemistry and mass losses: apply masks
            chem_loss_raw = criterion_chem(chem_preds, y_batch)
            mole_loss_raw = criterion_mole(mole_preds, m_batch)
            bulk_loss_raw = criterion_bulk(bulk_preds, x_batch[:,3:])
            #mole_zero_mask = (torch.sigmoid(logits) > 0.5).to(torch.float).detach() # Only use binary preds for masking when those neurons are free
            mole_zero_mask = (b_batch > 0.5).to(torch.float).detach()
     
            chem_loss_masked = (chem_loss_raw * chem_zero_mask * compWeights).sum() / (chem_zero_mask * compWeights).sum().clamp(min=1)
            mole_loss_masked = (mole_loss_raw * mole_zero_mask * binWeights).sum() / (mole_zero_mask * binWeights).sum().clamp(min=1)
            bulk_loss_masked = (bulk_loss_raw * bulk_zero_mask[:,3:]).sum() / (bulk_zero_mask[:,3:]).sum().clamp(min=1)
            
            running_sat_loss += loss_sat.item()
            running_mole_loss += mole_loss_masked.item()
            running_chem_loss += chem_loss_masked.item()
            running_bulk_loss += bulk_loss_masked.item()
            
            
            
            loss = loss_sat + chem_alpha*chem_loss_masked + mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
            #print(f"Sat loss: {loss_sat.item()}, Chem loss: {chem_loss_masked.item()}")

            if not torch.isfinite(loss):
                print("Non-finite loss detected!")
                print(f"Sat loss: {loss_sat.item()}, Chem loss: {chem_loss_masked.item()}, Mole loss: {mole_loss_masked}, Bulk loss: {bulk_loss_masked}")
                continue

            loss.backward()


            optimizer.step()

            running_train_loss += loss.item() * x_batch.size(0)

            if batch_idx % 200 == 0:
                percent_done = 100 * batch_idx / total_batches
                train_losses.append(loss.item())
                print(f"[{percent_done:>5.1f}%] Batch {batch_idx:>5d} Loss: {loss.item():.4f}")


        avg_train_loss = running_train_loss / len(full_train_set)

        print(f"Running Saturation Loss: {running_sat_loss/len(full_train_set):.3e}\nRunning Chem Loss: {running_chem_loss/len(full_train_set):.3e}")
        print(f"Running Molar Loss: {running_mole_loss/len(full_train_set):.3e}\nRunning Bulk Loss: {running_bulk_loss/len(full_train_set):.3e}")

        # ---- Evaluation ----
        FullMELTS.eval()
        running_test_loss = 0.0
        running_sat_loss = 0
        running_chem_loss = 0
        running_mole_loss = 0
        running_bulk_loss = 0
        
        with torch.no_grad():
            for batch_idx, (x_batch, b_batch, y_batch, m_batch) in enumerate(test_loader):
                x_batch, b_batch, y_batch, m_batch = x_batch.to(device, non_blocking=True), b_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True), m_batch.to(device, non_blocking=True)
                logits, chem_preds, chem_zero_mask, mole_preds, bulk_preds = FullMELTS(x_batch, binaries=b_batch, NN_only = True)
                
                # Binary saturation loss
                loss_sat = criterion_sat(logits, b_batch)
                
                # Chemistry losses: apply masks
                chem_loss_raw = criterion_chem(chem_preds, y_batch)
                mole_loss_raw = criterion_mole(mole_preds, m_batch)
                bulk_loss_raw = criterion_bulk(bulk_preds, x_batch[:,3:])
                bulk_zero_mask = (x_batch[:,3:] != 0).to(torch.float)
                #mole_zero_mask = (torch.sigmoid(logits) > 0.5).to(torch.float).detach()
                mole_zero_mask = (b_batch > 0.5).to(torch.float)
                
                chem_loss_masked = (chem_loss_raw * chem_zero_mask*compWeights).sum() / (chem_zero_mask*compWeights).sum().clamp(min=1)
                mole_loss_masked = (mole_loss_raw * mole_zero_mask*binWeights).sum() / (mole_zero_mask*binWeights).sum().clamp(min=1)
                bulk_loss_masked = (bulk_loss_raw * bulk_zero_mask).sum() / bulk_zero_mask.sum().clamp(min=1)
                
                running_sat_loss += loss_sat.item()
                running_mole_loss += mole_loss_masked.item()
                running_chem_loss += chem_loss_masked.item()
                running_bulk_loss += bulk_loss_masked.item()
                
                # Total loss
                #loss = loss_sat + chem_alpha*chem_loss_masked
                #loss = mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
                loss = loss_sat + chem_alpha*chem_loss_masked + mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
                

                running_test_loss += loss.item() * x_batch.size(0)
                if batch_idx % 50 == 0:
                    long_test_losses.append(loss.item())


        print(f"Running Saturation Loss: {running_sat_loss/len(full_test_set):.3e}\nRunning Chem Loss: {running_chem_loss/len(full_test_set):.3e}")
        print(f"Running Molar Loss: {running_mole_loss/len(full_test_set):.3e}\nRunning Bulk Loss: {running_bulk_loss/len(full_test_set):.3e}")

        
        avg_test_loss = running_test_loss / len(full_test_set)
        test_losses.append(avg_test_loss)
        if avg_test_loss <= valid_loss_min:
            #torch.save(FullMELTS.state_dict(), DictFilePath)
            FullMELTS.save(DictFilePath)
            print('\tValidation loss decreased ({:.6f} --> {:.6f}).  Saving model ...'.format(valid_loss_min, avg_test_loss))
            valid_loss_min = avg_test_loss
            improve_record.append(1)
        else:
            improve_record.append(0)

        print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.6f} | Test Loss: {avg_test_loss:.6f}")#/{EPOCHS}
        print(f"[TIMER] Epoch time: {time.time() - epoch_start:.2f} seconds")
        if np.sum(improve_record[-4:]) == 0:
            improve_record = [1,1,1,1]
            lr = lrs.pop()
            #if wd > 5*lr:
            #    wd = 5*lr
            print(f"No Improvement in 4 epochs. New LR: {lr}. New Weight Decay: {wd}")
            optimizer = AdamW(filter(lambda p: p.requires_grad, FullMELTS.parameters()), lr=lr, weight_decay=wd)#Adam(FullMELTS.parameters(), lr=lr)#, weight_decay=wd)
        #Adam(FullMELTS.parameters(), lr=lr)#, weight_decay=wd)

        #ADAPTIVE DROPOUT
        if avg_test_loss > avg_train_loss * 1.02:
            anyDropout = False
            if min(dropout_rate + 0.05, 0.6) > dropout_rate:
                old_drop = dropout_rate
                dropout_rate = min(dropout_rate + 0.05, 0.6)
                for module in FullMELTS.modules():
                    if isinstance(module, nn.Dropout):
                        module.p = dropout_rate
                        anyDropout  = True
                if anyDropout:
                    print(f"Overfitting. Increasing Dropout: {old_drop} -> {dropout_rate}")
            else: 
                print(f"Overfitting, but dropout_rate is at the maximum: {dropout_rate}")


        elif avg_test_loss < avg_train_loss:
            anyDropout = False
            if max(dropout_rate - 0.02, 0) < dropout_rate:
                old_drop = dropout_rate
                dropout_rate = max(dropout_rate - 0.02, 0) 
                for module in FullMELTS.modules():
                    if isinstance(module, nn.Dropout):
                        module.p = dropout_rate
                        anyDropout  = True
                if anyDropout:
                    print(f"Underfitting. Decreasing Dropout: {old_drop}->{dropout_rate}")

            else:
                print(f"Underfitting, but dropout_rate is at the minimum: {dropout_rate}")

        
        #break

In [ ]:
#Full L2 head polishing. 
#torch.autograd.set_detect_anomaly(False)

date = 'Nov13'

weight_dict_both = {
    'olivine': 3,
    'nepheline': 15,
    'leucite': 10,
    'alloy-solid': 5,
    'muscovite': 3,
    'k-feldspar': 5
}
# This overprints the above weights, does not apply to NoCr model
weight_dict_Cr = {
    'rhm-oxide': 5
}

binWeightsNoCr = torch.ones(len(list(label_indices.keys())), dtype = torch.float32).reshape(1,-1)
binWeightsCr = torch.ones(len(list(label_indices.keys())), dtype = torch.float32).reshape(1,-1)
compWeightsNoCr = torch.ones(label_indices_comp['melts-liquid'][-1]+1, dtype = torch.float32).reshape(1,-1)
compWeightsCr = torch.ones(label_indices_comp['melts-liquid'][-1]+1, dtype = torch.float32).reshape(1,-1)
"""for phase, W in weight_dict_both.items(): # Weights removed for 110 due to bad overfitting... Take a look at phase abundances??? 
    binWeightsCr[:,mass_phasedict[phase]] = W
    binWeightsNoCr[:,mass_phasedict[phase]] = W
    if phase in compositionally_variable_phases:
        compWeightsCr[:,comp_phasedict[phase]] = W
        compWeightsNoCr[:,comp_phasedict[phase]] = W
for phase, W in weight_dict_Cr.items():
    binWeightsCr[:,mass_phasedict[phase]] = W
    if phase in compositionally_variable_phases:
        compWeightsCr[:,comp_phasedict[phase]] = W"""


for i, (full_train_set, full_test_set) in enumerate([(full_train_set_NoCr, full_test_set_NoCr), (full_train_set_Cr, full_test_set_Cr)]):
    #if i == 0:
    #    continue
    
    """FullMELTS = NN.MidLevelNetwork(
            encoderLayerUp=1,
            encoderLayerDown=0,
            middleLayerUp=2,
            middleLayerDown=1,
            low_regularization='layernorm',
            high_regularization='dropout0.2'
        ).to(device).cuda()"""
    
    #DualSaturationChemistry().cuda()
    #date = "Sept30"
    #modelname = "rhyoliteMELTS1.0.2Fxtal"
    ##DictFilePath=f'./{modelname}_ChemMoleL2_{date}.pt'
    #date = "Oct16"
    DictFilePath=f"Models/MELTS{MELTSModel}{CalcType}{['NoCr','Cr'][i]}_FullL1_{date}.pt"
    #DictFilePath=f"Models/{modelname}{['NoCr', 'Cr'][i]}_ChemMoleL2_{date}.pt"

    FullMELTS = rebuild_MELTS_model(DictFilePath)

    #DictFilePath=f"Models/{modelname}{['NoCr', 'Cr'][i]}_BinaryOnly_{date}.pt"
    #FullMELTS.load_state_dict(torch.load(DictFilePath),strict = False)
    

    #date = "Sept22" + ['NoCr', 'Cr'][i]
    #date = 'Oct11'
    DictFilePath=f"Models/{modelname}{['NoCr', 'Cr'][i]}_FullL2_{date}.pt"
    
    Reg = FullMELTS.config['high_regularization']    
    wd = FullMELTS.config['highWD']
    noise = FullMELTS.config['noise']

    if 'dropout' in Reg.lower():
        dropout_rate = pull_number(Reg)
    else:
        dropout_rate = 0
    
    
    batch_size = 1024

    device = 'cuda'

    binWeights = [binWeightsNoCr, binWeightsCr][i]
    binWeights = binWeights.to(device)
    compWeights = [compWeightsNoCr, compWeightsCr][i]
    compWeights = (compWeights).to(device)
    FullMELTS = FullMELTS.cuda()
    
    train_loader = DataLoader(full_train_set, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    test_loader = DataLoader(full_test_set, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    # All but mole heads: Put those on top.
    for p in FullMELTS.parameters():
        p.requires_grad = True
    #for p in FullMELTS.mole_head.parameters():
    #    p.requires_grad = False
    """for p in FullMELTS.encoder.parameters():
        p.requires_grad = False
    for p in FullMELTS.sat_head.parameters():
        p.requires_grad = False"""
        
        
    criterion_sat = torch.nn.BCEWithLogitsLoss(weight = binWeights)
    criterion_chem = symmetric_rel_l2 #relative_L1_loss #y_pred, y_true, mask=None, eps=1e-6 #criterion_chem = F.mse_loss 
    criterion_mole = symmetric_rel_l2
    criterion_bulk = symmetric_rel_l2
    
    chem_alpha = 1#100 # Oct 16: moved from 1->100 due to two order of magnitude difference between saturation and other losses 
    mole_alpha = 1#100
    bulk_alpha = 1#50
    

    train_losses = []
    test_losses = []
    long_test_losses = []
    valid_loss_min = np.inf


    epoch = 0
    improve_record = [1,1]
    lrs = np.logspace(-7,-3,9).tolist()
    lr = lrs.pop()
    wd = FullMELTS.highWD # Oct 16: moved 1E-5 -> 1E-4 was 1E-5 for MELTS102 before forcing superliquidus to regurgitate inputs 
    optimizer = AdamW(filter(lambda p: p.requires_grad, FullMELTS.parameters()), lr=lr, weight_decay=wd)

    # ---- Training loop ----
    #for epoch in range(EPOCHS):
    while (lr > 2E-7):
    #while lr > 2E-25:

        epoch += 1
        epoch_start = time.time()
        FullMELTS.train()
        running_train_loss = 0.0
        total_batches = len(train_loader)

        running_sat_loss = 0
        running_chem_loss = 0
        running_mole_loss = 0
        running_bulk_loss = 0

        print(f"\n--- Epoch {epoch} ---") #/{EPOCHS}

        for batch_idx, (x_batch, b_batch, y_batch, m_batch) in enumerate(tqdm(train_loader, desc="Training", leave=False)):

            optimizer.zero_grad()

            x_batch, b_batch, y_batch, m_batch = x_batch.to(device, non_blocking=True), b_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True), m_batch.to(device, non_blocking=True)
            
            bulk_zero_mask = (x_batch != 0).to(torch.float) # Bulk zero mask different shape for training and testing because this mask is doubling as a filter for the noise
            #x_batch = x_batch + torch.randn_like(x_batch) * 0.005 * bulk_zero_mask # Add Small Gaussian Noise to avoid overfitting during training Oct 14: Moved 0.002->0.005
            if noise != 0:
                x_batch = x_batch + (x_batch * torch.randn_like(x_batch) * noise)
            logits, chem_preds, chem_zero_mask, mole_preds, bulk_preds = FullMELTS(x_batch, binaries=b_batch, NN_only = True)

            # Binary saturation loss
            loss_sat = criterion_sat(logits, b_batch) 

            # Chemistry and mass losses: apply masks
            chem_loss_raw = criterion_chem(chem_preds, y_batch)
            mole_loss_raw = criterion_mole(mole_preds, m_batch)
            bulk_loss_raw = criterion_bulk(bulk_preds, x_batch[:,3:])
            #mole_zero_mask = (torch.sigmoid(logits) > 0.5).to(torch.float).detach() # Only use binary preds for masking when those neurons are free
            mole_zero_mask = (b_batch > 0.5).to(torch.float).detach()
     
            chem_loss_masked = (chem_loss_raw * chem_zero_mask * compWeights).sum() / (chem_zero_mask * compWeights).sum().clamp(min=1)
            mole_loss_masked = (mole_loss_raw * mole_zero_mask * binWeights).sum() / (mole_zero_mask * binWeights).sum().clamp(min=1)
            bulk_loss_masked = (bulk_loss_raw * bulk_zero_mask[:,3:]).sum() / (bulk_zero_mask[:,3:]).sum().clamp(min=1)
            
            running_sat_loss += loss_sat.item()
            running_mole_loss += mole_loss_masked.item()
            running_chem_loss += chem_loss_masked.item()
            running_bulk_loss += bulk_loss_masked.item()
            
            
            
            loss = loss_sat + chem_alpha*chem_loss_masked + mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
            #print(f"Sat loss: {loss_sat.item()}, Chem loss: {chem_loss_masked.item()}")

            if not torch.isfinite(loss):
                print("Non-finite loss detected!")
                print(f"Sat loss: {loss_sat.item()}, Chem loss: {chem_loss_masked.item()}, Mole loss: {mole_loss_masked}, Bulk loss: {bulk_loss_masked}")
                continue

            loss.backward()


            optimizer.step()

            running_train_loss += loss.item() * x_batch.size(0)

            if batch_idx % 200 == 0:
                percent_done = 100 * batch_idx / total_batches
                train_losses.append(loss.item())
                print(f"[{percent_done:>5.1f}%] Batch {batch_idx:>5d} Loss: {loss.item():.4f}")


        avg_train_loss = running_train_loss / len(full_train_set)

        print(f"Running Saturation Loss: {running_sat_loss/len(full_train_set):.3e}\nRunning Chem Loss: {running_chem_loss/len(full_train_set):.3e}")
        print(f"Running Molar Loss: {running_mole_loss/len(full_train_set):.3e}\nRunning Bulk Loss: {running_bulk_loss/len(full_train_set):.3e}")

        # ---- Evaluation ----
        FullMELTS.eval()
        running_test_loss = 0.0
        running_sat_loss = 0
        running_chem_loss = 0
        running_mole_loss = 0
        running_bulk_loss = 0
        
        with torch.no_grad():
            for batch_idx, (x_batch, b_batch, y_batch, m_batch) in enumerate(test_loader):
                x_batch, b_batch, y_batch, m_batch = x_batch.to(device, non_blocking=True), b_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True), m_batch.to(device, non_blocking=True)
                logits, chem_preds, chem_zero_mask, mole_preds, bulk_preds = FullMELTS(x_batch, binaries=b_batch, NN_only = True)
                
                # Binary saturation loss
                loss_sat = criterion_sat(logits, b_batch)
                
                # Chemistry losses: apply masks
                chem_loss_raw = criterion_chem(chem_preds, y_batch)
                mole_loss_raw = criterion_mole(mole_preds, m_batch)
                bulk_loss_raw = criterion_bulk(bulk_preds, x_batch[:,3:])
                bulk_zero_mask = (x_batch[:,3:] != 0).to(torch.float)
                #mole_zero_mask = (torch.sigmoid(logits) > 0.5).to(torch.float).detach()
                mole_zero_mask = (b_batch > 0.5).to(torch.float)
                
                chem_loss_masked = (chem_loss_raw * chem_zero_mask*compWeights).sum() / (chem_zero_mask*compWeights).sum().clamp(min=1)
                mole_loss_masked = (mole_loss_raw * mole_zero_mask*binWeights).sum() / (mole_zero_mask*binWeights).sum().clamp(min=1)
                bulk_loss_masked = (bulk_loss_raw * bulk_zero_mask).sum() / bulk_zero_mask.sum().clamp(min=1)
                
                running_sat_loss += loss_sat.item()
                running_mole_loss += mole_loss_masked.item()
                running_chem_loss += chem_loss_masked.item()
                running_bulk_loss += bulk_loss_masked.item()
                
                # Total loss
                #loss = loss_sat + chem_alpha*chem_loss_masked
                #loss = mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
                loss = loss_sat + chem_alpha*chem_loss_masked + mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
                

                running_test_loss += loss.item() * x_batch.size(0)
                if batch_idx % 50 == 0:
                    long_test_losses.append(loss.item())


        print(f"Running Saturation Loss: {running_sat_loss/len(full_test_set):.3e}\nRunning Chem Loss: {running_chem_loss/len(full_test_set):.3e}")
        print(f"Running Molar Loss: {running_mole_loss/len(full_test_set):.3e}\nRunning Bulk Loss: {running_bulk_loss/len(full_test_set):.3e}")

        
        avg_test_loss = running_test_loss / len(full_test_set)
        test_losses.append(avg_test_loss)
        if avg_test_loss <= valid_loss_min:
            #torch.save(FullMELTS.state_dict(), DictFilePath)
            FullMELTS.save(DictFilePath)
            print('\tValidation loss decreased ({:.6f} --> {:.6f}).  Saving model ...'.format(valid_loss_min, avg_test_loss))
            valid_loss_min = avg_test_loss
            improve_record.append(1)
        else:
            improve_record.append(0)

        print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.6f} | Test Loss: {avg_test_loss:.6f}")#/{EPOCHS}
        print(f"[TIMER] Epoch time: {time.time() - epoch_start:.2f} seconds")
        if np.sum(improve_record[-3:]) == 0:
            improve_record = [1,1,1]
            lr = lrs.pop()
            #if wd > 5*lr:
            #    wd = 5*lr
            print(f"No Improvement in 3 epochs. New LR: {lr}. New Weight Decay: {wd}")
            optimizer = AdamW(filter(lambda p: p.requires_grad, FullMELTS.parameters()), lr=lr, weight_decay=wd)#Adam(FullMELTS.parameters(), lr=lr)#, weight_decay=wd)
        #Adam(FullMELTS.parameters(), lr=lr)#, weight_decay=wd)

        #ADAPTIVE DROPOUT
        if avg_test_loss > avg_train_loss * 1.02:
            anyDropout = False
            if min(dropout_rate + 0.05, 0.6) > dropout_rate:
                old_drop = dropout_rate
                dropout_rate = min(dropout_rate + 0.05, 0.6)
                for module in FullMELTS.modules():
                    if isinstance(module, nn.Dropout):
                        module.p = dropout_rate
                        anyDropout  = True
                if anyDropout:
                    print(f"Overfitting. Increasing Dropout: {old_drop} -> {dropout_rate}")
            else: 
                print(f"Overfitting, but dropout_rate is at the maximum: {dropout_rate}")


        elif avg_test_loss < avg_train_loss:
            anyDropout = False
            if max(dropout_rate - 0.02, 0) < dropout_rate:
                old_drop = dropout_rate
                dropout_rate = max(dropout_rate - 0.02, 0) 
                for module in FullMELTS.modules():
                    if isinstance(module, nn.Dropout):
                        module.p = dropout_rate
                        anyDropout  = True
                if anyDropout:
                    print(f"Underfitting. Decreasing Dropout: {old_drop}->{dropout_rate}")

            else:
                print(f"Underfitting, but dropout_rate is at the minimum: {dropout_rate}")

        
        #break

In [ ]:
#Chem and Mole head L2 polishing. (WEIGHTS DEACTIVATED) NoBulk, frozen encodings
#torch.autograd.set_detect_anomaly(False)

weight_dict_both = {
    'olivine': 3,
    'nepheline': 15,
    'leucite': 10,
    'alloy-solid': 5,
    'muscovite': 3,
    'k-feldspar': 5
}
# This overprints the above weights, does not apply to NoCr model
weight_dict_Cr = {
    'rhm-oxide': 5
}

binWeightsNoCr = torch.ones(len(list(label_indices.keys())), dtype = torch.float32).reshape(1,-1)
binWeightsCr = torch.ones(len(list(label_indices.keys())), dtype = torch.float32).reshape(1,-1)
compWeightsNoCr = torch.ones(label_indices_comp['melts-liquid'][-1]+1, dtype = torch.float32).reshape(1,-1)
compWeightsCr = torch.ones(label_indices_comp['melts-liquid'][-1]+1, dtype = torch.float32).reshape(1,-1)
"""for phase, W in weight_dict_both.items():
    binWeightsCr[:,mass_phasedict[phase]] = W
    binWeightsNoCr[:,mass_phasedict[phase]] = W
    if phase in compositionally_variable_phases:
        compWeightsCr[:,comp_phasedict[phase]] = W
        compWeightsNoCr[:,comp_phasedict[phase]] = W
for phase, W in weight_dict_Cr.items():
    binWeightsCr[:,mass_phasedict[phase]] = W
    if phase in compositionally_variable_phases:
        compWeightsCr[:,comp_phasedict[phase]] = W"""


for i, (full_train_set, full_test_set) in enumerate([(full_train_set_NoCr, full_test_set_NoCr), (full_train_set_Cr, full_test_set_Cr)]):

    #FullMELTS = DualSaturationChemistry().cuda()
    #date = "Sept30" 
    #modelname = "rhyoliteMELTS1.0.2Fxtal"
    DictFilePath=f"Models/{modelname}{['NoCr', 'Cr'][i]}_FullL2_{date}.pt"
    
    #FullMELTS.load_state_dict(torch.load(DictFilePath),strict = False)
    FullMELTS = rebuild_MELTS_model(DictFilePath)
    #date = "Sept22"
    DictFilePath=f"Models/{modelname}{['NoCr', 'Cr'][i]}_Final_{date}.pt"
    
    Reg = FullMELTS.config['high_regularization']    
    wd = FullMELTS.config['highWD']
    noise = FullMELTS.config['noise']

    if 'dropout' in Reg.lower():
        dropout_rate = pull_number(Reg)
    else:
        dropout_rate = 0
    
    
    batch_size = 1024

    device = 'cuda'

    binWeights = [binWeightsNoCr, binWeightsCr][i]
    binWeights = binWeights.to(device)
    compWeights = [compWeightsNoCr, compWeightsCr][i]
    compWeights = (compWeights).to(device)
    FullMELTS = FullMELTS.cuda()
    
    train_loader = DataLoader(full_train_set, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    test_loader = DataLoader(full_test_set, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    # All but mole heads: Put those on top.
    for p in FullMELTS.parameters():
        p.requires_grad = True
    #for p in FullMELTS.mole_head.parameters():
    #    p.requires_grad = False
    for p in FullMELTS.encoder.parameters():
        p.requires_grad = False
    for p in FullMELTS.middleBrain.parameters():
        p.requires_grad = False
        
    criterion_sat = torch.nn.BCEWithLogitsLoss(weight = binWeights)
    criterion_chem = symmetric_rel_l2 #relative_L1_loss #y_pred, y_true, mask=None, eps=1e-6 #criterion_chem = F.mse_loss 
    criterion_mole = symmetric_rel_l2
    criterion_bulk = symmetric_rel_l2
    
    chem_alpha = 1#100
    mole_alpha = 1#100
    bulk_alpha = 0 # For now, guassian noise in bulk
    

    train_losses = []
    test_losses = []
    long_test_losses = []
    valid_loss_min = np.inf


    epoch = 0
    improve_record = [1,1,1,1]
    lrs = np.logspace(-7,-4,2).tolist()
    lr = lrs.pop()
    wd = FullMELTS.highWD
    optimizer = AdamW(filter(lambda p: p.requires_grad, FullMELTS.parameters()), lr=lr, weight_decay=wd)

    # ---- Training loop ----
    #for epoch in range(EPOCHS):
    while (lr > 2E-7):
    #while lr > 2E-25:

        epoch += 1
        epoch_start = time.time()
        FullMELTS.train()
        running_train_loss = 0.0
        total_batches = len(train_loader)

        print(f"\n--- Epoch {epoch} ---") #/{EPOCHS}

        for batch_idx, (x_batch, b_batch, y_batch, m_batch) in enumerate(tqdm(train_loader, desc="Training", leave=False)):

            optimizer.zero_grad()

            x_batch, b_batch, y_batch, m_batch = x_batch.to(device, non_blocking=True), b_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True), m_batch.to(device, non_blocking=True)
            
            bulk_zero_mask = (x_batch != 0).to(torch.float)
            #x_batch = x_batch + torch.randn_like(x_batch) * 0.002 * bulk_zero_mask # Add Small Gaussian Noise to avoid overfitting during training
            if noise != 0:
                x_batch = x_batch + (x_batch * torch.randn_like(x_batch) * noise)
                
            logits, chem_preds, chem_zero_mask, mole_preds, bulk_preds = FullMELTS(x_batch, binaries=b_batch, NN_only = True)

            # Binary saturation loss
            loss_sat = criterion_sat(logits, b_batch)

            # Chemistry and mass losses: apply masks
            chem_loss_raw = criterion_chem(chem_preds, y_batch)
            mole_loss_raw = criterion_mole(mole_preds, m_batch)
            bulk_loss_raw = criterion_bulk(bulk_preds, x_batch[:,3:])
            mole_zero_mask = (torch.sigmoid(logits) > 0.5).to(torch.float).detach() # Only use binary preds for masking when those neurons are free
            mole_zero_mask = (b_batch > 0.5).to(torch.float).detach()
     
            chem_loss_masked = (chem_loss_raw * chem_zero_mask * compWeights).sum() / (chem_zero_mask * compWeights).sum().clamp(min=1)
            mole_loss_masked = (mole_loss_raw * mole_zero_mask * binWeights).sum() / (mole_zero_mask * binWeights).sum().clamp(min=1)
            bulk_loss_masked = (bulk_loss_raw * bulk_zero_mask[:,3:]).sum() / (bulk_zero_mask[:,3:]).sum().clamp(min=1)
            
            
            loss = loss_sat + chem_alpha*chem_loss_masked + mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
            #print(f"Sat loss: {loss_sat.item()}, Chem loss: {chem_loss_masked.item()}")

            if not torch.isfinite(loss):
                print("Non-finite loss detected!")
                print(f"Sat loss: {loss_sat.item()}, Chem loss: {chem_loss_masked.item()}, Mole loss: {mole_loss_masked}, Bulk loss: {bulk_loss_masked}")
                continue

            loss.backward()


            optimizer.step()

            running_train_loss += loss.item() * x_batch.size(0)

            if batch_idx % 200 == 0:
                percent_done = 100 * batch_idx / total_batches
                train_losses.append(loss.item())
                print(f"[{percent_done:>5.1f}%] Batch {batch_idx:>5d} Loss: {loss.item():.4f}")


        avg_train_loss = running_train_loss / len(full_train_set)

        # ---- Evaluation ----
        FullMELTS.eval()
        running_test_loss = 0.0
        running_sat_loss = 0
        running_chem_loss = 0
        running_mole_loss = 0
        running_bulk_loss = 0
        
        with torch.no_grad():
            for batch_idx, (x_batch, b_batch, y_batch, m_batch) in enumerate(test_loader):
                x_batch, b_batch, y_batch, m_batch = x_batch.to(device, non_blocking=True), b_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True), m_batch.to(device, non_blocking=True)
                logits, chem_preds, chem_zero_mask, mole_preds, bulk_preds = FullMELTS(x_batch, binaries=b_batch, NN_only = True)
                
                # Binary saturation loss
                loss_sat = criterion_sat(logits, b_batch)
                running_sat_loss += loss_sat.item()
                
                # Chemistry losses: apply masks
                chem_loss_raw = criterion_chem(chem_preds, y_batch)
                mole_loss_raw = criterion_mole(mole_preds, m_batch)
                bulk_loss_raw = criterion_bulk(bulk_preds, x_batch[:,3:])
                bulk_zero_mask = (x_batch[:,3:] != 0).to(torch.float)
                mole_zero_mask = (torch.sigmoid(logits) > 0.5).to(torch.float).detach()
                #mole_zero_mask = (b_batch > 0.5).to(torch.float)
                
                chem_loss_masked = (chem_loss_raw * chem_zero_mask).sum() / chem_zero_mask.sum().clamp(min=1)
                mole_loss_masked = (mole_loss_raw * mole_zero_mask).sum() / mole_zero_mask.sum().clamp(min=1)
                bulk_loss_masked = (bulk_loss_raw * bulk_zero_mask).sum() / bulk_zero_mask.sum().clamp(min=1)
                
                running_mole_loss += mole_loss_masked.item()
                running_chem_loss += chem_loss_masked.item()
                running_bulk_loss += bulk_loss_masked.item()
                
                # Total loss
                #loss = loss_sat + chem_alpha*chem_loss_masked
                #loss = mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
                loss = loss_sat + chem_alpha*chem_loss_masked + mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
                

                running_test_loss += loss.item() * x_batch.size(0)
                if batch_idx % 50 == 0:
                    long_test_losses.append(loss.item())


        print(f"Running Saturation Loss: {round(running_sat_loss,4)}\nRunning Chem Loss: {round(running_chem_loss,4)}")
        print(f"Running Molar Loss: {round(running_mole_loss,4)}\nRunning Bulk Loss: {round(running_bulk_loss,4)}")


        avg_test_loss = running_test_loss / len(full_test_set)
        test_losses.append(avg_test_loss)
        if avg_test_loss <= valid_loss_min:
            #torch.save(FullMELTS.state_dict(), DictFilePath)
            FullMELTS.save(DictFilePath)
            print('\tValidation loss decreased ({:.6f} --> {:.6f}).  Saving model ...'.format(valid_loss_min, avg_test_loss))
            valid_loss_min = avg_test_loss
            improve_record.append(1)
        else:
            improve_record.append(0)

        print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.6f} | Test Loss: {avg_test_loss:.6f}")#/{EPOCHS}
        print(f"[TIMER] Epoch time: {time.time() - epoch_start:.2f} seconds")
        if np.sum(improve_record[-3:]) == 0:
            improve_record = [1,1,1,1]
            lr = lrs.pop()
            #if wd > 5*lr:
            #    wd = 5*lr
            print(f"No Improvement in 3 epochs. New LR: {lr}. New Weight Decay: {wd}")
            optimizer = AdamW(filter(lambda p: p.requires_grad, FullMELTS.parameters()), lr=lr, weight_decay=wd)#Adam(FullMELTS.parameters(), lr=lr)#, weight_decay=wd)
        #Adam(FullMELTS.parameters(), lr=lr)#, weight_decay=wd)

        #ADAPTIVE DROPOUT
        if avg_test_loss > avg_train_loss * 1.02:
            anyDropout = False
            if min(dropout_rate + 0.05, 0.6) > dropout_rate:
                old_drop = dropout_rate
                dropout_rate = min(dropout_rate + 0.05, 0.6)
                for module in FullMELTS.modules():
                    if isinstance(module, nn.Dropout):
                        module.p = dropout_rate
                        anyDropout  = True
                if anyDropout:
                    print(f"Overfitting. Increasing Dropout: {old_drop} -> {dropout_rate}")
            else: 
                print(f"Overfitting, but dropout_rate is at the maximum: {dropout_rate}")


        elif avg_test_loss < avg_train_loss:
            anyDropout = False
            if max(dropout_rate - 0.02, 0) < dropout_rate:
                old_drop = dropout_rate
                dropout_rate = max(dropout_rate - 0.02, 0) 
                for module in FullMELTS.modules():
                    if isinstance(module, nn.Dropout):
                        module.p = dropout_rate
                        anyDropout  = True
                if anyDropout:
                    print(f"Underfitting. Decreasing Dropout: {old_drop}->{dropout_rate}")

            else:
                print(f"Underfitting, but dropout_rate is at the minimum: {dropout_rate}")

        
        #break

In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
import time
import itertools
import gc
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
criterion = nn.BCEWithLogitsLoss()

# === Range of layer depths to test ===
depth_range = range(0, 5)  # 0, 1, 2, 3, 4

#results = []

# === Loop over combinations of (encoderUp, encoderDown, middleUp, middleDown) ===
#for enc_up, enc_down, mid_up, mid_down in itertools.product(depth_range, depth_range, depth_range, depth_range):
for enc_up, enc_down in itertools.product(depth_range, depth_range):#, depth_range, depth_range):
    if enc_up >= enc_down:
        
        print(f"\n==============================")
        print(f"Testing configuration: encoderUp={enc_up}, encoderDown={enc_down}, middleUp={mid_up}, middleDown={mid_down}")
        print(f"==============================")

        # --- Build model ---
        model = MidLevelNetwork(
            encoderLayerUp=enc_up,
            encoderLayerDown=enc_down,
            middleLayerUp=0,
            middleLayerDown=0,
            regularization='none'
        ).to(device)

        # freeze chem & mole heads like original code
        for p in model.parameters():
            p.requires_grad = True
        for p in model.chem_heads.parameters():
            p.requires_grad = False
        for p in model.mole_head.parameters():
            p.requires_grad = False

        # --- Loaders (same for both "Cr" and "NoCr" if you only want one test here) ---
        batch_size = 1024
        train_loader = DataLoader(binary_train_set_NoCr, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
        test_loader = DataLoader(binary_test_set_NoCr, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

        optimizer = Adam(model.parameters(), lr=1e-4, weight_decay=0)
        best_test_loss = np.inf
        train_losses, test_losses = [], []

        # --- Train for up to 10 epochs ---
        for epoch in range(10):
            start = time.time()
            model.train()
            running_train_loss = 0.0

            for xb, yb in tqdm(train_loader, desc=f"Train Epoch {epoch+1}", leave=False):
                xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
                xb = xb + torch.randn_like(xb) * 0.0025  # noise injection

                optimizer.zero_grad()
                logits = model.forward_binaries(xb)
                loss = criterion(logits, yb)
                loss.backward()
                optimizer.step()

                running_train_loss += loss.item() * xb.size(0)

            avg_train_loss = running_train_loss / len(train_loader.dataset)
            train_losses.append(avg_train_loss)

            # --- Evaluate ---
            model.eval()
            running_test_loss = 0.0
            with torch.no_grad():
                for xb, yb in test_loader:
                    xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
                    logits = model.forward_binaries(xb)
                    loss = criterion(logits, yb)
                    running_test_loss += loss.item() * xb.size(0)

            avg_test_loss = running_test_loss / len(test_loader.dataset)
            test_losses.append(avg_test_loss)

            print(f"Epoch {epoch+1:02d}: Train {avg_train_loss:.5f} | Test {avg_test_loss:.5f} | Δt={time.time()-start:.1f}s")

            # simple early stopping
            if avg_test_loss < best_test_loss:
                best_test_loss = avg_test_loss
            elif epoch > 2 and avg_test_loss > best_test_loss * 1.01:
                print("No improvement; stopping early.")
                break

            gc.collect()

        # --- Record result for this configuration ---
        results.append({
            'encoderUp': enc_up,
            'encoderDown': enc_down,
            'train_loss': train_losses[-1],
            'test_loss': test_losses[-1]
        })

# --- Print results summary ---
print("\n=== Final Results Summary ===")
for r in results:
    print(f"Enc({r['encoderUp']},{r['encoderDown']})) "
          f"=> Train={r['train_loss']:.5f}  Test={r['test_loss']:.5f}")

# --- Optional: visualize losses ---
plt.figure(figsize=(8,6))
plt.scatter([f"E{r['encoderUp']}{r['encoderDown']}" for r in results],
            [r['test_loss'] for r in results], c='red', label='Test Loss')
plt.xticks(rotation=90)
plt.ylabel("Loss")
plt.title("Model Depth vs Test Loss (10 epochs)")
plt.tight_layout()
plt.legend()
plt.show()


In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
import time
import itertools
import gc
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
criterion = nn.BCEWithLogitsLoss()

# === Range of layer depths to test ===
#depth_range = range(0, 5)  # 0, 1, 2, 3, 4
depth_range = [0]
regs = ['Dropout0.1', 'Dropout0.05']
#results[0]['regularization'] = 'none'
#results = []

# === Loop over combinations of (encoderUp, encoderDown, middleUp, middleDown) ===
#for enc_up, enc_down, mid_up, mid_down in itertools.product(depth_range, depth_range, depth_range, depth_range):
for enc_up, enc_down, reg in itertools.product(depth_range, depth_range, regs):#, depth_range, depth_range):
    if enc_up >= enc_down:
        
        print(f"\n==============================")
        print(f"Testing configuration: encoderUp={enc_up}, encoderDown={enc_down}, reg = {reg}")
        print(f"==============================")

        # --- Build model ---
        model = MidLevelNetwork(
            encoderLayerUp=enc_up,
            encoderLayerDown=enc_down,
            middleLayerUp=0,
            middleLayerDown=0,
            regularization=reg
        ).to(device)

        # freeze chem & mole heads like original code
        for p in model.parameters():
            p.requires_grad = True
        for p in model.chem_heads.parameters():
            p.requires_grad = False
        for p in model.mole_head.parameters():
            p.requires_grad = False

        # --- Loaders (same for both "Cr" and "NoCr" if you only want one test here) ---
        batch_size = 1024
        train_loader = DataLoader(binary_train_set_NoCr, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
        test_loader = DataLoader(binary_test_set_NoCr, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

        optimizer = Adam(model.parameters(), lr=1e-4, weight_decay=0)
        best_test_loss = np.inf
        train_losses, test_losses = [], []

        # --- Train for up to 10 epochs ---
        for epoch in range(10):
            start = time.time()
            model.train()
            running_train_loss = 0.0

            for xb, yb in tqdm(train_loader, desc=f"Train Epoch {epoch+1}", leave=False):
                xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
                xb = xb + torch.randn_like(xb) * 0.0025  # noise injection

                optimizer.zero_grad()
                logits = model.forward_binaries(xb)
                loss = criterion(logits, yb)
                loss.backward()
                optimizer.step()

                running_train_loss += loss.item() * xb.size(0)

            avg_train_loss = running_train_loss / len(train_loader.dataset)
            train_losses.append(avg_train_loss)

            # --- Evaluate ---
            model.eval()
            running_test_loss = 0.0
            with torch.no_grad():
                for xb, yb in test_loader:
                    xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
                    logits = model.forward_binaries(xb)
                    loss = criterion(logits, yb)
                    running_test_loss += loss.item() * xb.size(0)

            avg_test_loss = running_test_loss / len(test_loader.dataset)
            test_losses.append(avg_test_loss)

            print(f"Epoch {epoch+1:02d}: Train {avg_train_loss:.5f} | Test {avg_test_loss:.5f} | Δt={time.time()-start:.1f}s")

            # simple early stopping
            if avg_test_loss < best_test_loss:
                best_test_loss = avg_test_loss
            elif epoch > 2 and avg_test_loss > best_test_loss * 1.01:
                print("No improvement; stopping early.")
                break

            gc.collect()

        # --- Record result for this configuration ---
        results.append({
            'encoderUp': enc_up,
            'encoderDown': enc_down,
            'train_loss': train_losses[-1],
            'test_loss': test_losses[-1],
            'regularization': reg
        })

# --- Print results summary ---
print("\n=== Final Results Summary ===")
for r in results:
    print(f"Enc({r['encoderUp']},{r['encoderDown']})) "
          f"=> Train={r['train_loss']:.5f}  Test={r['test_loss']:.5f}")

# --- Optional: visualize losses ---
plt.figure(figsize=(8,6))
plt.scatter([f"E{r['encoderUp']}{r['encoderDown']}" for r in results],
            [r['test_loss'] for r in results], c='red', label='Test Loss')
plt.xticks(rotation=90)
plt.ylabel("Loss")
plt.title("Model Depth vs Test Loss (10 epochs)")
plt.tight_layout()
plt.legend()
plt.show()


In [ ]:
from shutil import RegistryError
from webbrowser import WindowsDefault
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
import time
import itertools
import gc
import matplotlib.pyplot as plt

import importlib
import BackEnds.nnMELTS as NN
importlib.reload(NN)

device = 'cuda' #if torch.cuda.is_available() else 'cpu'
criterion = nn.BCEWithLogitsLoss()

# === Range of layer depths to test ===
depth_range = range(2)  # 0, 1, 2, 3, 4
results = []
#WDs = [1E-2, 1, 0]
#noises = [0, 0.005, 0.01]
WDs = [0.1,0.25,1]#[0.1,1] 
noises = [0,0.002]
Regs = ['layernormdropout0']#, 'batchnorm', 'none'] # Now include dropout for 
enc_up = 1
enc_down = 1
noise = 0
max_epochs = 10
# === Loop over combinations of (encoderUp, encoderDown, middleUp, middleDown) ===
#for enc_up, enc_down, mid_up, mid_down in itertools.product(depth_range, depth_range, depth_range, depth_range):
#for enc_up, enc_down, WD, Reg in itertools.product(depth_range, depth_range, WDs, Regs):#, depth_range, depth_range):
#for  WD, Reg in itertools.product(WDs, Regs):#, depth_range, depth_range):

#for WD in WDs:#, depth_range, depth_range):
for WD, noise, Reg in itertools.product(WDs, noises, Regs):#, depth_range, depth_range):
    if 'dropout' in Reg.lower():
        dropout_rate = pull_number(Reg)
    else:
        dropout_rate = 0

    if enc_up >= enc_down and (enc_up + enc_down) > 0:
        
        print(f"\n==============================")
        print(f"Testing configuration: encoderUp={enc_up}, encoderDown={enc_down}, WD = {WD}, noise = {noise}, Regularization: {Reg}")
        print(f"==============================")

        # --- Build model ---
        model = NN.MidLevelNetwork(
            encoderLayerUp=enc_up,
            encoderLayerDown=enc_down,
            middleLayerUp=0,
            middleLayerDown=0,
            low_regularization=Reg
        ).to(device)

        # freeze chem & mole heads like original code
        for p in model.parameters():
            p.requires_grad = True
        for p in model.chem_heads.parameters():
            p.requires_grad = False
        for p in model.mole_head.parameters():
            p.requires_grad = False
        if model.middleBrain is not None:
            for p in model.middleBrain.parameters():
                p.requires_grad = False

        # --- Loaders (same for both "Cr" and "NoCr" if you only want one test here) ---
        batch_size = 1024
        train_loader = DataLoader(binary_train_set_NoCr, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
        test_loader = DataLoader(binary_test_set_NoCr, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

        optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=WD)
        best_test_loss = np.inf
        train_losses, test_losses = [], []

        # --- Train for limited epochs ---
        for epoch in range(max_epochs):
            start = time.time()
            model.train()
            running_train_loss = 0.0

            for xb, yb in tqdm(train_loader, desc=f"Train Epoch {epoch+1}", leave=False):
                xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
                xb = xb + torch.randn_like(xb) * noise  # noise injection

                optimizer.zero_grad()
                logits = model.forward_binaries(xb)
                loss = criterion(logits, yb)
                loss.backward()
                optimizer.step()

                running_train_loss += loss.item() * xb.size(0)

            avg_train_loss = running_train_loss / len(train_loader.dataset)
            train_losses.append(avg_train_loss)

            # --- Evaluate ---
            model.eval()
            running_test_loss = 0.0
            with torch.no_grad():
                for xb, yb in test_loader:
                    xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
                    logits = model.forward_binaries(xb)
                    loss = criterion(logits, yb)
                    running_test_loss += loss.item() * xb.size(0)

            avg_test_loss = running_test_loss / len(test_loader.dataset)
            test_losses.append(avg_test_loss)

            print(f"Epoch {epoch+1:02d}: Train {avg_train_loss:.5f} | Test {avg_test_loss:.5f} | Δt={time.time()-start:.1f}s")

            # simple early stopping
            if avg_test_loss < best_test_loss:
                best_test_loss = avg_test_loss
                torch.save(model.state_dict(),'Oct20Binary110ParamSearch.pt')
            elif epoch > 2 and avg_test_loss > best_test_loss :
                print("No improvement; stopping early.")
                break

            gc.collect()

            #ADAPTIVE DROPOUT
            if avg_test_loss > avg_train_loss * 1.02 and 'dropout' in Reg.lower():
                anyDropout = False
                if min(dropout_rate + 0.05, 0.6) != dropout_rate:
                    for module in model.modules():
                        if isinstance(module, nn.Dropout):
                            module.p = dropout_rate
                            anyDropout  = True
                    if anyDropout:
                        print(f"Overfitting. Increasing Dropout: {dropout_rate}->{min(dropout_rate + 0.05, 0.6)}") # Only print when the model is actually changed
                else: 
                    print(f"Overfitting, but dropout_rate is at the maximum: {dropout_rate}")


            elif avg_test_loss < avg_train_loss and 'dropout' in Reg.lower():
                anyDropout = False
                if max(dropout_rate - 0.02, 0) != dropout_rate:
                    for module in model.modules():
                        if isinstance(module, nn.Dropout):
                            module.p = dropout_rate
                            anyDropout  = True
                    if anyDropout:
                        print(f"Underfitting. Decreasing Dropout: {dropout_rate}->{max(dropout_rate - 0.02, 0)}")

                else:
                    print(f"Underfitting, but dropout_rate is at the minimum: {dropout_rate}")
            

        # --- Record result for this configuration ---
        results.append({
            'encoderUp': enc_up,
            'encoderDown': enc_down,
            'train_loss': train_losses[-1],
            'test_loss': test_losses[-1],
            'wd':{WD},
            'reg':{Reg}
        })

# --- Print results summary ---
print("\n=== Final Results Summary ===")
for r in results:
    print(f"Enc({r['encoderUp']},{r['encoderDown']},WD:{r['wd']},LowReg:{r['reg']})) "
          f"=> Train={r['train_loss']:.5f}  Test={r['test_loss']:.5f}")

# --- Optional: visualize losses ---
plt.figure(figsize=(8,6))
plt.scatter([f"E{r['encoderUp']}{r['encoderDown']}" for r in results],
            [r['test_loss'] for r in results], c='red', label='Test Loss')
plt.xticks(rotation=90)
plt.ylabel("Loss")
plt.title("Model Depth vs Test Loss (10 epochs)")
plt.tight_layout()
plt.legend()
plt.show()


In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
import time
import gc
import itertools
import matplotlib.pyplot as plt
import importlib

import BackEnds.nnMELTS as NN
importlib.reload(NN)


date = "Oct22"
checkpoint = torch.load(f"Models/MELTS{MELTSModel}{CalcType}{['NoCr', 'Cr'][i]}_BinaryOnlyLayerNorm_{date}.pt", map_location='cpu')

binWeightsNoCr = torch.ones(len(list(label_indices.keys())), dtype = torch.float32).reshape(1,-1)
binWeightsCr = torch.ones(len(list(label_indices.keys())), dtype = torch.float32).reshape(1,-1)
compWeightsNoCr = torch.ones(label_indices_comp['melts-liquid'][-1]+1, dtype = torch.float32).reshape(1,-1)
compWeightsCr = torch.ones(label_indices_comp['melts-liquid'][-1]+1, dtype = torch.float32).reshape(1,-1)

device = 'cuda' #if torch.cuda.is_available() else 'cpu'

# === Define your search range ===
middle_up_range = [0,1,2]
middle_down_range = [0,1,2]
regs = ['none']#, 'dropout0.2', 'layernorm', 'batchnorm', 'dropout0.4']
results = []

# === Loop over NoCr and Cr datasets ===
for i, (full_train_set, full_test_set) in enumerate([
    (full_train_set_NoCr, full_test_set_NoCr)#,
    #(full_train_set_Cr, full_test_set_Cr)
]):

    model_tag = ['NoCr', 'Cr'][i]
    binWeights = [binWeightsNoCr, binWeightsCr][i]
    binWeights.to(device)
    compWeights = [compWeightsNoCr, compWeightsCr][i]
    compWeights.to(device)

    train_loader = DataLoader(full_train_set, batch_size=1024, shuffle=True, num_workers=4, pin_memory=True)
    test_loader = DataLoader(full_test_set, batch_size=1024, shuffle=False, num_workers=4, pin_memory=True)
    wd = 0
    # === Sweep middle layer parameters ===
    for mid_up, mid_down, reg in itertools.product(middle_up_range, middle_down_range, regs):  
        if mid_up >= mid_down:
            print(f"\n{'='*60}")
            print(f"Training FullMELTS-{model_tag} | middleUp={mid_up}, middleDown={mid_down}, regularization = {reg}, WD = {wd}")
            print(f"{'='*60}")

            # --- Build model ---
            FullMELTS = NN.MidLevelNetwork(
                encoderLayerUp=1,
                encoderLayerDown=0,
                middleLayerUp=mid_up,
                middleLayerDown=mid_down,
                low_regularization='layernorm',
                high_regularization= reg
            ).to(device)

            

            # Get the current model's parameters
            model_dict = FullMELTS.state_dict()

            # --- Filter keys to include only encoder and phase head layers ---
            # Adjust these substrings to match your actual module names.
            allowed_prefixes = [
                "encoder.",            # everything under encoder
                "sat_head."        # or "binaries_head." / "chem_heads." if named that way
            ]

            filtered_dict = {
                k: v for k, v in checkpoint.items()
                if any(k.startswith(p) for p in allowed_prefixes)
            }

            # --- Update and load only matching weights ---
            model_dict.update(filtered_dict)
            FullMELTS.load_state_dict(model_dict, strict=False)

            DictFilePath = f"Models/{modelname}{model_tag}_midU{mid_up}_midD{mid_down}_{date}.pt"

            # freeze chem & mole heads like original code
            for p in FullMELTS.parameters():
                p.requires_grad = True
            for p in FullMELTS.encoder.parameters():
                p.requires_grad = False
            for p in FullMELTS.sat_head.parameters():
                p.requires_grad = False

            # === Set up loss functions ===
            criterion_sat = nn.BCEWithLogitsLoss(weight=binWeights)
            criterion_chem = symmetric_rel_l2
            criterion_mole = symmetric_rel_l2
            criterion_bulk = symmetric_rel_l2

            sat_alpha = 0
            chem_alpha = 1
            mole_alpha = 1
            bulk_alpha = 0

            # === Optimizer and scheduler ===
            lr = 1e-3
            wd = 0
            optimizer = AdamW(FullMELTS.parameters(), lr=lr, weight_decay=wd)

            # === Training ===
            best_loss = np.inf
            improve_record = [1, 1]
            #lrs = np.logspace(-7, -4, 2).tolist()
            lrs = []
            epoch = 0
            test_losses = []
            train_losses = []

            
            running_sat_loss = 0
            running_chem_loss = 0
            running_mole_loss = 0
            running_bulk_loss = 0

            while epoch < 5:#2E-7:
                epoch += 1
                FullMELTS.train()
                running_train_loss = 0.0

                print(f"\n--- Epoch {epoch} ---")

                for batch_idx, (x_batch, b_batch, y_batch, m_batch) in enumerate(tqdm(train_loader, desc=f"{model_tag} Train", leave=False)):
                    optimizer.zero_grad()

                    x_batch, b_batch, y_batch, m_batch = (
                        x_batch.to(device, non_blocking=True),
                        b_batch.to(device, non_blocking=True),
                        y_batch.to(device, non_blocking=True),
                        m_batch.to(device, non_blocking=True)
                    )

                    bulk_zero_mask = (x_batch != 0).float()
                    #x_batch = x_batch + torch.randn_like(x_batch) * 0.003 * bulk_zero_mask

                    logits, chem_preds, chem_zero_mask, mole_preds, bulk_preds = FullMELTS(x_batch, binaries=b_batch, NN_only=True)
                    
                    loss_sat = criterion_sat(logits, b_batch)
                    chem_loss_raw = criterion_chem(chem_preds, y_batch)
                    mole_loss_raw = criterion_mole(mole_preds, m_batch)
                    bulk_loss_raw = criterion_bulk(bulk_preds, x_batch[:, 3:])

                    mole_zero_mask = (torch.sigmoid(logits) > 0.5).float().detach()
                    chem_loss_masked = (chem_loss_raw * chem_zero_mask * compWeights).sum() / (chem_zero_mask * compWeights).sum().clamp(min=1)
                    mole_loss_masked = (mole_loss_raw * mole_zero_mask * binWeights).sum() / (mole_zero_mask * binWeights).sum().clamp(min=1)
                    bulk_loss_masked = (bulk_loss_raw * bulk_zero_mask).sum() / bulk_zero_mask.sum().clamp(min=1)

                    running_sat_loss += loss_sat
                    running_chem_loss += chem_loss_masked
                    running_mole_loss += mole_loss_masked
                    running_bulk_loss += bulk_loss_masked

                    loss = sat_alpha * loss_sat + chem_alpha * chem_loss_masked + mole_alpha * mole_loss_masked + bulk_alpha * bulk_loss_masked
                    if not torch.isfinite(loss):
                        print("Non-finite loss detected — skipping batch.")
                        print(f"Running Saturation Loss: {loss_sat}\nRunning Chem Loss: {chem_loss_masked}")
                        print(f"Running Molar Loss: {mole_loss_masked}\nRunning Bulk Loss: {bulk_loss_masked}")
                        continue

                    loss.backward()
                    optimizer.step()
                    running_train_loss += loss.item() * x_batch.size(0)

                
                print(f"TRAIN Running Saturation Loss: {running_sat_loss/ len(full_train_set)}\nRunning Chem Loss: {running_chem_loss/ len(full_train_set)}")
                print(f"Running Molar Loss: {running_mole_loss/ len(full_train_set)}\nRunning Bulk Loss: {running_bulk_loss/ len(full_train_set)}")
                avg_train_loss = running_train_loss / len(full_train_set)
                train_losses.append(avg_train_loss)

                # === Validation ===
                FullMELTS.eval()
                running_test_loss = 0.0

                running_sat_loss = 0
                running_chem_loss = 0
                running_mole_loss = 0
                running_bulk_loss = 0

                with torch.no_grad():
                    for x_batch, b_batch, y_batch, m_batch in test_loader:
                        x_batch, b_batch, y_batch, m_batch = (
                            x_batch.to(device, non_blocking=True),
                            b_batch.to(device, non_blocking=True),
                            y_batch.to(device, non_blocking=True),
                            m_batch.to(device, non_blocking=True)
                        )

                        logits, chem_preds, chem_zero_mask, mole_preds, bulk_preds = FullMELTS(x_batch, binaries=b_batch, NN_only=True)

                        loss_sat = criterion_sat(logits, b_batch)
                        chem_loss_raw = criterion_chem(chem_preds, y_batch)
                        mole_loss_raw = criterion_mole(mole_preds, m_batch)
                        bulk_loss_raw = criterion_bulk(bulk_preds, x_batch[:, 3:])

                        mole_zero_mask = (torch.sigmoid(logits) > 0.5).float().detach()
                        chem_loss_masked = (chem_loss_raw * chem_zero_mask).mean()
                        mole_loss_masked = (mole_loss_raw * mole_zero_mask).mean()
                        bulk_loss_masked = (bulk_loss_raw).mean()

                        running_sat_loss += loss_sat
                        running_chem_loss += chem_loss_masked
                        running_mole_loss += mole_loss_masked
                        running_bulk_loss += bulk_loss_masked

                        loss = sat_alpha * loss_sat + chem_alpha * chem_loss_masked + mole_alpha * mole_loss_masked + bulk_alpha * bulk_loss_masked
                        running_test_loss += loss.item() * x_batch.size(0)

                avg_test_loss = running_test_loss / len(full_test_set)
                test_losses.append(avg_test_loss)

                print(f"TEST Running Saturation Loss: {running_sat_loss/ len(full_test_set)}\nRunning Chem Loss: {running_chem_loss/ len(full_test_set)}")
                print(f"Running Molar Loss: {running_mole_loss/ len(full_test_set)}\nRunning Bulk Loss: {running_bulk_loss/ len(full_test_set)}")

                print(f"Epoch {epoch} | Train {avg_train_loss:.6f} | Test {avg_test_loss:.6f}")

                if avg_test_loss < best_loss:
                    best_loss = avg_test_loss
                    torch.save(FullMELTS.state_dict(), DictFilePath)
                    print(f"↓ Improved to {avg_test_loss:.6f}, model saved.")
                    improve_record.append(1)
                else:
                    improve_record.append(0)

                if np.sum(improve_record[-2:]) == 0:
                    improve_record = [1, 1]
                    if lrs:
                        lr = lrs.pop()
                        print(f"No improvement. New LR: {lr}")
                        optimizer = AdamW(FullMELTS.parameters(), lr=lr, weight_decay=wd)
                    else:
                        break

                gc.collect()

            # Record result
            results.append({
                'model': model_tag,
                'upperReg': reg,
                'middleUp': mid_up,
                'middleDown': mid_down,
                'best_test_loss': best_loss
            })

# === Summary ===
print("\n=== Summary of Middle Layer Tests ===")
for r in results:
    print(f"{r['model']}: midU={r['middleUp']} midD={r['middleDown']} → test_loss={r['best_test_loss']:.6f}")

plt.figure(figsize=(8,6))
plt.scatter(
    [f"{r['model']}_U{r['middleUp']}D{r['middleDown']}" for r in results],
    [r['best_test_loss'] for r in results],
    c=['red' if r['model']=='Cr' else 'blue' for r in results]
)
plt.xticks(rotation=90)
plt.ylabel("Test Loss")
plt.title("Middle Layer Depth vs Test Loss (NoCr vs Cr)")
plt.tight_layout()
plt.show()


In [ ]:
# === Summary ===
print("\n=== Summary of Middle Layer Tests ===")
for r in results:
    print(f"{r['model']}: midU={r['middleUp']} midD={r['middleDown']} → test_loss={r['best_test_loss']:.6f}")

plt.figure(figsize=(8,6))
plt.scatter(
    [f"{r['model']}_U{r['middleUp']}D{r['middleDown']}" for r in results],
    [r['best_test_loss'] for r in results],
    c=['red' if r['model']=='Cr' else 'blue' for r in results]
)
plt.xticks(rotation=90)
plt.ylabel("Test Loss")
plt.title("Middle Layer Depth vs Test Loss (NoCr vs Cr)")
plt.tight_layout()
plt.show()
